# 알약 검출 데이터 EDA

## EDA 목적

- 모델을 학습하기 전에 대회 데이터의 구성, 라벨 품질, 클래스 분포, bbox 분포, 조합과 촬영 각도, test 데이터를 확인한다.
- 확인한 결과를 바탕으로 라벨 정제, train/val 분할, 입력 크기, 데이터 증강, 보충 데이터에 대해 정할 것을 정리한다.
- 대회 데이터만으로 부족한 점을 확인한다.

## 데이터

- 대회 데이터(Kaggle): `train_images`, `train_annotations`, `test_images`

## 목차

| 섹션 | 내용 |
|---|---|
| 0. 사전 설정 | 라이브러리, 경로, 한글 글꼴, seed |
| 1. 데이터 구성 | 폴더 구조, 파일 수와 용량, 파일명 규칙, JSON 구조와 필드, 이미지와 JSON의 연결 |
| 2. 이미지 | train/test 샘플, 이미지 크기, 촬영 조건, train과 test 비교, 중복 이미지 |
| 3. 라벨 품질 | 라벨 누락, 이미지 밖 bbox, bbox 겹침, 클래스별 bbox 영역 모아 보기, area 불일치, 결측치, 오류 목록과 고치는 방안 |
| 4. 클래스 | category_id와 약 이름의 대응, 클래스 분포, 클래스별 조합 수, 외형 정보 |
| 5. bbox 분포 | bbox 크기와 위치, 입력 크기에 주는 영향 |
| 6. 조합과 촬영 각도 | 조합당 사진 수, 같은 조합 사진 비교, 분할 단위에 주는 의미 |
| 7. test 데이터 | test에만 있는 클래스와 그에 따른 한계 |
| 8. 1차 학습에 적용한 라벨 수정과 train/val 분할 | 라벨 수정 목록, train/val 분할 |

## 실행 방법

- 대회 데이터 압축을 `data/raw` 아래에 풀어 둔다. `train_images`, `train_annotations`, `test_images` 폴더가 있으면 된다.
- 처음부터 순서대로 실행한다. 뒤 섹션은 앞 섹션에서 만든 변수를 쓴다.
- 원본 데이터는 읽기만 하고 고치지 않는다.

## 0. 사전 설정

뒤 섹션에서 공통으로 쓰는 것을 먼저 준비한다.

| 단계 | 하는 일 | 이유 |
|---|---|---|
| 0-1 | 라이브러리 불러오기 | 여러 섹션에서 쓰는 라이브러리를 한곳에서 불러온다. |
| 0-2 | 경로 설정 | 노트북을 어느 폴더에서 열어도, 압축을 어떤 방식으로 풀었어도 같은 데이터를 찾도록 한다. |
| 0-3 | 한글 글꼴 설정 | 그래프의 제목과 축 이름을 한글로 쓰면 matplotlib 기본 글꼴에서는 네모로 깨진다. |
| 0-4 | seed 고정 | 샘플을 무작위로 뽑거나 train/val을 나눠 볼 때, 다시 실행하거나 다른 사람이 실행해도 같은 결과가 나오게 한다. |

여러 섹션에서 쓰는 도우미 함수는 여기서 미리 만들지 않고, 처음 필요한 섹션에서 만든다.

### 0-1. 라이브러리 불러오기

In [ ]:
import json
import platform
import sys
from collections import Counter
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager
from PIL import Image

# 실행 환경을 남겨 둔다. 결과가 다르게 나올 때 버전 차이부터 확인하기 위해서다.
print("Python    :", sys.version.split()[0])
print("numpy     :", np.__version__)
print("pandas    :", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("Pillow    :", Image.__version__)

### 0-2. 경로 설정

- 노트북 위치와 상관없이 같은 경로를 쓰도록, `pyproject.toml`이 있는 폴더를 레포 최상위 폴더로 잡는다.
- 원본 폴더는 `data/raw` 아래에서 `train_images` 폴더가 있는 곳으로 정한다. 압축을 푼 방식에 따라 `data/raw` 바로 아래일 수도 있고, 한 단계 아래 폴더일 수도 있기 때문이다.
- 경로는 내 컴퓨터의 절대경로 대신 레포 최상위 기준 상대경로로 출력한다.

In [ ]:
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
assert ROOT is not None, "pyproject.toml이 있는 레포 폴더를 찾지 못했습니다. 레포 안에서 노트북을 실행하세요."


def rel(path):
    """레포 최상위 기준 상대경로를 / 구분자로 돌려준다. (내 컴퓨터의 절대경로를 출력하지 않기 위해)"""
    return Path(path).relative_to(ROOT).as_posix()


# 원본: data/raw 아래에서 train_images 폴더가 있는 곳
found = [p for p in sorted((ROOT / "data" / "raw").rglob("train_images")) if p.is_dir()]
assert len(found) == 1, f"data/raw 아래에 train_images 폴더가 1개여야 합니다. 찾은 수: {len(found)}"
RAW_DIR = found[0].parent

TRAIN_IMG_DIR = RAW_DIR / "train_images"
TRAIN_ANN_DIR = RAW_DIR / "train_annotations"
TEST_IMG_DIR = RAW_DIR / "test_images"

for folder in [TRAIN_IMG_DIR, TRAIN_ANN_DIR, TEST_IMG_DIR]:
    assert folder.is_dir(), f"폴더가 없습니다: {rel(folder)}. 대회 데이터 압축을 data/raw 아래에 풀었는지 확인하세요."

print("레포 최상위 :", ROOT.name)
print("원본 폴더   :", rel(RAW_DIR))
print("train 이미지:", rel(TRAIN_IMG_DIR))
print("train 라벨  :", rel(TRAIN_ANN_DIR))
print("test 이미지 :", rel(TEST_IMG_DIR))

### 0-3. 한글 글꼴 설정

- 운영체제마다 기본으로 있는 한글 글꼴이 다르다. Windows는 맑은 고딕, macOS는 AppleGothic이 있고, 둘 다 없으면 나눔고딕을 찾는다.
- 한글 글꼴로 바꾸면 음수 기호(-)가 네모로 깨질 수 있어서, 음수 기호는 일반 하이픈으로 그리도록 설정한다.

In [ ]:
KOREAN_FONTS = ["Malgun Gothic", "AppleGothic", "NanumGothic"]
installed = {f.name for f in font_manager.fontManager.ttflist}
FONT = next((name for name in KOREAN_FONTS if name in installed), None)
assert FONT is not None, f"한글 글꼴을 찾지 못했습니다. {KOREAN_FONTS} 중 하나를 설치한 뒤 커널을 다시 시작하세요."

plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = False  # 한글 글꼴에서 음수 기호가 깨지지 않도록

print("운영체제:", platform.system(), "/ 글꼴:", FONT)

### 0-4. seed 고정

seed 값만 정한다. 무작위로 뽑거나 나누는 셀은 셀 안에서 `np.random.default_rng(SEED)`로 새로 만들어 쓴다. 셀을 다시 실행해도 같은 결과가 나오게 하기 위해서다.

In [ ]:
SEED = 42  # 무작위로 뽑거나 나누는 셀에서 np.random.default_rng(SEED)로 쓴다

print("SEED:", SEED)

## 1. 데이터 구성

데이터가 어떤 파일로 이루어져 있고, 라벨(JSON)이 이미지와 어떻게 연결되는지 확인한다. 이 섹션에서 만드는 어노테이션 표(`ann_df`)는 뒤 섹션에서 계속 쓴다.

| 단계 | 하는 일 |
|---|---|
| 1-1 | 대회가 설명한 데이터 구조 정리 |
| 1-2 | 폴더 구조와 파일 수, 용량 확인 |
| 1-3 | 파일명 규칙 확인 |
| 1-4 | JSON 하나 열어 보기 |
| 1-5 | 전체 JSON 구조 확인 |
| 1-6 | 어노테이션 표(`ann_df`) 만들기 |
| 1-7 | 이미지와 JSON의 연결 확인 |

라벨이 맞는지(라벨 누락, 좌표 오류 등)는 3. 라벨 품질에서 확인한다. 이 섹션은 구조만 본다.

### 1-1. 대회 제공 데이터 설명

대회 데이터 설명(레포의 `DATA.md`)에서 이 EDA에 필요한 부분만 정리한다. 실제 데이터가 이 설명과 같은지는 1-2부터 확인한다. 전체 필드 목록은 `DATA.md`에 있다.

#### 파일 구성

| 폴더 | 설명 |
|---|---|
| `train_images` | 훈련 이미지(PNG) |
| `train_annotations` | 훈련 라벨(COCO 형식 JSON) |
| `test_images` | 테스트 이미지(PNG). 라벨이 없다. |

- 원본은 AI Hub 경구약제 이미지 데이터이고, 대회 측이 가공해서 제공했다.
- test 데이터에는 train에 없는 클래스가 있고, 제공된 데이터에는 크고 작은 오류가 들어 있다고 안내되어 있다.

#### JSON 구조

JSON은 `images`, `annotations`, `categories` 세 목록과 `type`으로 되어 있다. 이 EDA에서 주로 쓰는 항목은 아래와 같다.

| 구분 | 항목 | 설명 |
|---|---|---|
| `images[]` | `id`, `file_name`, `width`, `height` | 이미지 식별자, 파일명, 너비, 높이 |
| `images[]` | `drug_N`, `dl_idx`, `dl_name` | 알약 코드, 알약 식별자, 제품명 |
| `images[]` | `camera_la`, `camera_lo`, `size`, `drug_dir`, `back_color`, `light_color`, `drug_S` | 카메라 위도, 카메라 경도, 이미지 사이즈, 알약 방향, 촬영 배경, 촬영 조명, 알약 상태 |
| `images[]` | `drug_shape`, `color_class1`, `print_front`, `print_back` 등 | 알약 모양, 색상, 식별 문자 등 외형 정보 |
| `annotations[]` | `id`, `image_id`, `category_id` | 어노테이션 식별자, 이미지 식별자, 클래스 |
| `annotations[]` | `bbox`, `area` | bbox 좌표 `[x, y, w, h]`(왼쪽 위 꼭짓점과 너비, 높이), 면적 |
| `categories[]` | `id`, `name` | 클래스 번호, 클래스 이름 |

#### 대회 측 가공 내역

원본에서는 `image_id`, `category_id`, annotation `id`가 모두 1이었는데, 대회 측이 아래처럼 다시 부여했다.

- `image_id`: 이미지 파일명마다 중복 없이 순서대로 부여
- `category_id`: JSON 안의 첫 번째 이미지에 있는 `dl_idx` 값을 정수로 바꿔 부여
- annotation `id`: 유효한 어노테이션에만 순서대로 부여. `bbox` 키가 있고 비어 있지 않으며(존재), 길이 4인 리스트(유효)여야 한다.

### 1-2. 폴더 구조와 파일 수, 용량

- 폴더 트리로 구조를 본다. 폴더 이름 옆의 `[폴더 n개 · 파일 n개]`는 바로 아래 폴더 수와 그 아래 전체 파일 수다. 한 폴더 안에서는 앞의 3개 항목만 보여 주고 나머지는 개수로 줄인다.
- 폴더별로 파일 수, 확장자, 용량, 0바이트 파일 수를 센다. PNG와 JSON이 아닌 파일이나 빈 파일이 섞여 있으면 여기서 드러난다.
- 마지막으로 뒤에서 쓸 파일 목록(train 이미지, test 이미지, JSON)을 만든다.

In [ ]:
def folder_summary(folder):
    """폴더 옆에 붙일 요약: [바로 아래 폴더 수 · 하위 전체 파일 수]"""
    n_dirs = sum(1 for p in folder.iterdir() if p.is_dir())
    n_files = sum(1 for p in folder.rglob("*") if p.is_file())
    return f"[폴더 {n_dirs}개 · 파일 {n_files}개]"


def show_tree(folder, max_depth=3, max_items=3, prefix=""):
    """폴더 구조를 트리 모양으로 출력한다. 한 폴더 안에서는 max_items개만 보여 주고 나머지는 개수로 줄인다."""
    children = sorted(folder.iterdir())
    dirs = [p for p in children if p.is_dir()]
    files = [p for p in children if p.is_file()]
    shown = (dirs + files)[:max_items]  # 폴더를 먼저, 파일을 나중에 보여 준다
    hidden_dirs = len(dirs) - sum(p.is_dir() for p in shown)
    hidden_files = len(files) - sum(p.is_file() for p in shown)
    has_hidden = hidden_dirs > 0 or hidden_files > 0

    for i, child in enumerate(shown):
        is_last = i == len(shown) - 1 and not has_hidden
        branch = "└── " if is_last else "├── "
        if child.is_dir():
            print(f"{prefix}{branch}{child.name}/  {folder_summary(child)}")
            if max_depth > 1:
                show_tree(child, max_depth - 1, max_items, prefix + ("    " if is_last else "│   "))
        else:
            print(f"{prefix}{branch}{child.name}")

    if has_hidden:
        rest = [f"폴더 {hidden_dirs}개"] * bool(hidden_dirs) + [f"파일 {hidden_files}개"] * bool(hidden_files)
        print(f"{prefix}└── ... 외 {', '.join(rest)}")


print(f"{rel(RAW_DIR)}/  {folder_summary(RAW_DIR)}")
show_tree(RAW_DIR)

In [ ]:
inventory = []
for folder in [TRAIN_IMG_DIR, TRAIN_ANN_DIR, TEST_IMG_DIR]:
    files = [p for p in folder.rglob("*") if p.is_file()]
    sizes = [p.stat().st_size for p in files]
    ext_counts = Counter(p.suffix.lower() or "(확장자 없음)" for p in files)
    inventory.append({
        "폴더": rel(folder),
        "파일 수": len(files),
        "확장자별 개수": ", ".join(f"{ext} {n}" for ext, n in sorted(ext_counts.items())),
        "용량(MiB)": round(sum(sizes) / 1024**2, 1),  # MiB = 1024 × 1024 바이트
        "0바이트 파일 수": sum(size == 0 for size in sizes),
    })
display(pd.DataFrame(inventory))

# 뒤에서 쓸 파일 목록. rglob은 하위 폴더가 몇 단계여도 모두 찾는다.
train_img_paths = sorted(TRAIN_IMG_DIR.rglob("*.png"))
test_img_paths = sorted(TEST_IMG_DIR.rglob("*.png"))
ann_paths = sorted(TRAIN_ANN_DIR.rglob("*.json"))

print(f"train 이미지: {len(train_img_paths)}장 / test 이미지: {len(test_img_paths)}장 / train JSON: {len(ann_paths)}개")

#### 1-2 결과

- `data/raw` 아래에 대회 데이터 세 폴더가 있다. 트리 마지막 줄의 파일 1개는 세 폴더 밖에 있는 파일(레포가 빈 폴더를 유지하려고 두는 `.gitkeep`으로 보인다)이라 분석에 쓰지 않는다.
- 폴더별 파일

| 폴더 | 파일 | 용량 |
|---|---|---|
| `train_images` | PNG 232장 | 387.7 MiB |
| `train_annotations` | JSON 763개 | 2.0 MiB |
| `test_images` | PNG 842장 | 1446.5 MiB |

- PNG, JSON 말고 다른 형식의 파일이나 0바이트 파일은 없다.
- `train_images`, `test_images`는 하위 폴더 없이 이미지가 바로 들어 있다.
- `train_annotations`는 3단계 구조다. `<조합 이름>_json` 폴더가 114개 있고, 그 안에 `K-<알약 코드>` 폴더가, 그 안에 이미지 파일명과 같은 이름의 JSON이 있다. 트리에 보인 조합 폴더들은 알약 코드 폴더 4개에 JSON 3개씩이었다.
- train 이미지는 232장인데 JSON은 763개라서, 이미지 한 장의 라벨이 JSON 여러 개에 나뉘어 있는 것으로 보인다. 1-5와 1-7에서 확인한다.

### 1-3. 파일명 규칙

- train 이미지 파일명을 `_`로 나눠, 자리마다 어떤 값이 몇 번 나오는지 센다. 값이 하나뿐인 자리와 여러 값이 있는 자리를 구분하려는 것이다. 각 자리가 무엇을 뜻하는지는 1-7에서 JSON 값과 비교해서 확인한다.
- 첫 번째 자리(첫 `_` 앞부분)는 `K-001900-016548-019607-029451`처럼 알약 코드를 `-`로 이어 붙인 형태다. 이 EDA에서는 이 부분을 조합 이름이라고 부른다. 함께 놓고 찍은 알약 묶음을 뜻하는 것으로 보이며, 조합 이름이 몇 종류인지와 알약 코드가 몇 개씩 들어 있는지 센다.
- test 이미지 파일명의 형식도 확인한다.

In [ ]:
def combo_name(file_name):
    """이미지 파일명에서 조합 이름(첫 _ 앞부분)을 꺼낸다. 예: K-001900-016548-019607-029451"""
    return file_name.split("_")[0]


def pill_codes(file_name):
    """조합 이름에 적힌 알약 코드를 정수 리스트로 꺼낸다. 예: K-001900-016548 → [1900, 16548]"""
    return [int(code) for code in combo_name(file_name).split("-")[1:]]


train_names = [p.name for p in train_img_paths]
print("예시:", train_names[0])
print("_로 나눈 부분 수 → 파일 수:", dict(Counter(len(Path(n).stem.split("_")) for n in train_names)))

# 자리별 값 분포. 0번 자리는 조합 이름이라 값 종류가 많아서 개수만 센다.
name_parts = pd.DataFrame([Path(n).stem.split("_") for n in train_names])
part_summary = []
for col in name_parts.columns:
    counts = name_parts[col].value_counts()
    part_summary.append({
        "자리": col,
        "값 종류 수": len(counts),
        "값 분포 (값: 파일 수)": "(조합 이름)" if col == 0 else ", ".join(f"{v}: {n}" for v, n in counts.items()),
    })
display(pd.DataFrame(part_summary))

combos = Counter(combo_name(n) for n in train_names)
print("조합 이름 종류:", len(combos))
print("조합 이름에 든 알약 코드 수 → 파일 수:", dict(sorted(Counter(len(pill_codes(n)) for n in train_names).items())))

# test 파일명: 확장자를 뺀 이름이 숫자인지 본다.
test_stems = [p.stem for p in test_img_paths]
non_digit = [s for s in test_stems if not s.isdigit()]
print()
print("test 예시:", [p.name for p in test_img_paths[:5]])
print("test 파일명 중 숫자가 아닌 것:", len(non_digit))
if not non_digit:
    numbers = sorted(int(s) for s in test_stems)
    print(f"test 번호 범위: {numbers[0]} ~ {numbers[-1]} (파일 {len(numbers)}개)")

#### 1-3 결과

- train 파일명 232개는 모두 `_`로 나누면 8부분이다.
- 0번 자리(조합 이름)는 114종류다. 조합 이름에 알약 코드가 3개인 파일이 157장, 4개인 파일이 75장이다.
- 값이 여러 개인 자리는 3번(0이 228장, 1이 4장)과 5번(90이 85장, 75가 75장, 70이 72장)뿐이다. 1, 2, 4, 6, 7번 자리는 232장 모두 같은 값(0, 2, 2, 000, 200)이다.
- 조합 이름은 114종류인데 파일은 232장이라, 같은 조합 이름의 사진이 여러 장 있다. 1-2 트리에 보인 첫 조합의 파일 3장은 5번 자리(70, 75, 90)만 달랐다. 조합별 사진 수는 6번에서 센다.
- test 파일명 842개는 모두 숫자이고 번호는 1~1500 사이다. 1500개 번호 중 842개만 있어서 번호가 이어져 있지 않다. test 파일명에는 조합 이름이나 촬영 정보가 없다.

### 1-4. JSON 하나 열어 보기

JSON 파일 하나를 열어 실제 형식이 1-1의 설명과 같은지 본다.

- 최상위 항목과 자료형, 목록마다 들어 있는 항목 수
- `annotations`, `categories`는 항목이 적어서 전체를 출력한다.
- `images[0]`은 항목이 많아서 표로 보여 준다(항목, 값, 자료형).
- Windows에서 인코딩을 지정하지 않고 열면 기본값(cp949)으로 읽어 한글 때문에 오류가 난다. 그래서 UTF-8로 연다. `utf-8-sig`는 파일 맨 앞에 BOM이 있어도 없어도 읽을 수 있는 방식이다.

In [ ]:
def read_json(path):
    """JSON 파일 하나를 읽어 dict로 돌려준다."""
    with open(path, encoding="utf-8-sig") as f:
        return json.load(f)


sample_path = ann_paths[0]
sample = read_json(sample_path)

print("파일:", sample_path.relative_to(TRAIN_ANN_DIR).as_posix())
print()
print("[최상위 항목]")
for key, value in sample.items():
    count = f"항목 {len(value)}개" if isinstance(value, list) else f"값 {value!r}"
    print(f"  {key}: {type(value).__name__}, {count}")

print()
print("[annotations]")
print(json.dumps(sample["annotations"], ensure_ascii=False, indent=2))  # ensure_ascii=False: 한글을 글자 그대로 출력
print("[categories]")
print(json.dumps(sample["categories"], ensure_ascii=False, indent=2))

image0 = sample["images"][0]
print()
print(f"[images[0]] 항목 {len(image0)}개")
display(pd.DataFrame({
    "항목": list(image0.keys()),
    "값": [str(v) for v in image0.values()],
    "자료형": [type(v).__name__ for v in image0.values()],
}))

#### 1-4 결과

- 최상위 항목은 `images`(목록, 1개), `type`(`'instances'`), `annotations`(목록, 1개), `categories`(목록, 1개)로, 1-1 설명과 같다.
- `annotations`에는 `area`, `iscrowd`, `bbox`, `category_id`, `ignore`, `segmentation`, `id`, `image_id` 8개 항목이 있다. `bbox`는 숫자 4개이고 `segmentation`은 빈 목록이다.
- `categories`에는 `supercategory`(`'pill'`), `id`, `name` 3개 항목이 있다. `id`(1900)는 `annotations`의 `category_id`와 같고, `name`은 제품명이다.
- `images[0]`에는 50개 항목이 있다. 대회 설명(`DATA.md`)의 49개 항목에 없는 `imgfile`이 하나 더 있다.
- 값이 빈 문자열인 항목이 여럿 있다(`di_company_mf`, `color_class2`, `line_front`, `line_back`, `mark_code_*` 등). 이 파일 하나의 결과이고, 전체에서 얼마나 비어 있는지는 3번에서 센다.
- 같은 번호인데 자료형이 다른 항목이 있다. `dl_idx`는 글자(`'1900'`), `category_id`는 정수(1900)다. 비교할 때는 자료형을 맞춰야 해서, 1-7에서는 숫자로 바꿔 비교한다.
- `images`의 `id`(34)와 `annotations`의 `image_id`(34)가 같다.

### 1-5. 전체 JSON 구조 확인

1-4는 파일 하나만 본 것이라, 모든 JSON이 같은 구조인지 확인한다.

- 모든 JSON을 한 번씩 읽어 `loaded`에 `(경로, 내용)`으로 담아 둔다. 뒤 셀은 파일을 다시 열지 않고 이 목록을 쓴다. 읽지 못한 파일은 멈추지 않고 따로 모아 보여 준다.
- JSON마다 `(images 수, annotations 수, categories 수)` 조합별로 파일 수를 센다.
- 최상위 항목 구성, 목록별 항목의 key 구성, `type` 값이 파일마다 같은지 센다.

In [ ]:
loaded = []  # (경로, JSON 내용)
failed = []  # 읽지 못한 파일
for path in ann_paths:
    try:
        loaded.append((path, read_json(path)))
    except (OSError, UnicodeDecodeError, json.JSONDecodeError) as e:
        failed.append({"파일": path.relative_to(TRAIN_ANN_DIR).as_posix(), "오류": str(e)})

print(f"읽기 성공: {len(loaded)}개 / 실패: {len(failed)}개")
if failed:
    display(pd.DataFrame(failed))

# JSON마다 (images 수, annotations 수, categories 수)
structure = Counter(
    (len(d.get("images", [])), len(d.get("annotations", [])), len(d.get("categories", [])))
    for _, d in loaded
)
display(pd.DataFrame(
    [{"images 수": i, "annotations 수": a, "categories 수": c, "JSON 수": n} for (i, a, c), n in sorted(structure.items())]
))

# 최상위 항목 구성과, 목록별 항목의 key 구성이 몇 가지인지
top_keys = Counter(tuple(sorted(d.keys())) for _, d in loaded)
print("최상위 항목 구성:", {", ".join(k): n for k, n in top_keys.items()})
for block in ["images", "annotations", "categories"]:
    key_sets = Counter(frozenset(item.keys()) for _, d in loaded for item in d.get(block, []))
    sizes = sorted({len(ks) for ks in key_sets})
    print(f"{block} 항목의 key 구성: {len(key_sets)}가지 (key 수: {sizes})")
print("type 값:", dict(Counter(d.get("type") for _, d in loaded)))

#### 1-5 결과

- JSON 763개를 모두 읽었고 실패는 없다.
- 763개 모두 `images`, `annotations`, `categories`가 1개씩이다. JSON 하나가 어노테이션 하나, 즉 알약 하나다.
- 최상위 항목 구성(`annotations`, `categories`, `images`, `type`)은 763개 모두 같다. 목록별 항목의 key 구성도 한 가지씩이다(`images` 50개, `annotations` 8개, `categories` 3개). `type`은 모두 `'instances'`다.

### 1-6. 어노테이션 표 만들기

모든 JSON을 어노테이션 하나가 한 행인 표 `ann_df`로 정리한다. 뒤 섹션의 분석은 이 표로 한다.

- 1-5에서 확인한 구조(JSON마다 `images`, `annotations`, `categories`가 몇 개인지)에 따라 표를 만든다. 아래 코드는 JSON마다 세 목록이 1개씩일 때를 전제로 하고, 아니면 멈춘다.
- 같은 정보가 여러 곳에 있는 항목(클래스 번호, 이미지 식별자 등)은 모두 열로 남겨서 1-7에서 서로 같은지 비교한다.

| 열 | 출처 | 설명 |
|---|---|---|
| `json_path` | 파일 경로 | `train_annotations` 기준 JSON 경로. 문제가 있는 행의 원본 파일을 찾을 때 쓴다. |
| `folder_code`, `combo` | 경로, 파일명 | 약 폴더 이름(`K-` 코드), 조합 이름 |
| `file_name`, `image_id`, `width`, `height` | `images[0]` | 이미지 파일명, 이미지 식별자, 너비, 높이 |
| `drug_N`, `dl_idx`, `dl_name` | `images[0]` | 알약 코드, 알약 식별자, 제품명 |
| `camera_la`, `camera_lo`, `size`, `drug_dir`, `back_color`, `light_color`, `drug_S` | `images[0]` | 촬영 정보 |
| `ann_id`, `ann_image_id`, `category_id` | `annotations[0]` | 어노테이션 식별자, 어노테이션이 가리키는 이미지 식별자, 클래스 |
| `bbox`, `bbox_x`, `bbox_y`, `bbox_w`, `bbox_h`, `area` | `annotations[0]` | bbox(원래 리스트와 값 4개), 면적 |
| `cat_id`, `cat_name` | `categories[0]` | 클래스 번호, 클래스 이름 |

In [ ]:
assert set(structure) == {(1, 1, 1)}, f"JSON마다 images, annotations, categories가 1개씩이 아닙니다: {dict(structure)}. 1-5 결과를 확인하세요."

IMAGE_COLUMNS = ["drug_N", "dl_idx", "dl_name", "camera_la", "camera_lo", "size",
                 "drug_dir", "back_color", "light_color", "drug_S"]

rows = []
for path, data in loaded:
    image, ann, cat = data["images"][0], data["annotations"][0], data["categories"][0]
    bbox = ann.get("bbox")
    bbox_ok = isinstance(bbox, list) and len(bbox) == 4  # 대회 설명의 유효 기준: 길이 4인 리스트
    x, y, w, h = bbox if bbox_ok else (None,) * 4
    rows.append({
        "json_path": path.relative_to(TRAIN_ANN_DIR).as_posix(),
        "folder_code": path.parent.name,
        "combo": combo_name(image["file_name"]),
        "file_name": image["file_name"],
        "image_id": image["id"],
        "width": image["width"],
        "height": image["height"],
        **{col: image.get(col) for col in IMAGE_COLUMNS},
        "ann_id": ann.get("id"),
        "ann_image_id": ann.get("image_id"),
        "category_id": ann.get("category_id"),
        "bbox": bbox,
        "bbox_x": x, "bbox_y": y, "bbox_w": w, "bbox_h": h,
        "area": ann.get("area"),
        "cat_id": cat.get("id"),
        "cat_name": cat.get("name"),
    })
ann_df = pd.DataFrame(rows)

print(f"표 크기: {ann_df.shape[0]}행 × {ann_df.shape[1]}열")
print(f"bbox가 길이 4인 리스트: {ann_df['bbox_x'].notna().sum()}개 / 아님: {ann_df['bbox_x'].isna().sum()}개")
display(ann_df.head())

#### 1-6 결과

- JSON 하나가 한 행인 763행 × 28열의 표를 만들었다.
- `bbox`는 763개 모두 길이 4인 리스트라 대회 설명의 유효 기준을 만족한다. 값이 이미지 안에 있는지, 실제 알약을 감싸는지는 3번에서 확인한다.

### 1-7. 이미지와 JSON의 연결

`ann_df`를 믿고 써도 되는지, 이미지 파일과 JSON, JSON 안의 여러 값이 서로 맞는지 확인한다.

| 검사 | 확인하는 것 |
|---|---|
| 이미지 파일과 JSON의 짝 | JSON이 없는 이미지(학습에 쓰면 그 안의 알약을 배경으로 배움), 이미지 파일이 없는 JSON |
| 이미지 한 장에 연결된 JSON 수 | JSON 하나가 알약 하나라면, 이미지 한 장의 라벨이 여러 JSON에 나뉘어 있다. |
| 이미지 식별자 | 같은 파일명은 같은 `image_id`인지, 같은 `image_id`가 다른 파일명에 쓰이지 않는지, `annotations`의 `image_id`가 같은 JSON의 `images`의 `id`와 같은지 |
| 같은 이미지의 크기 정보 | 같은 이미지의 JSON끼리 `width`, `height`가 같은지 |
| 클래스 번호 | 약 폴더 이름, `category_id`, `categories`의 `id`, `dl_idx`, `drug_N`이 모두 같은 번호인지, `categories`의 `name`과 `dl_name`이 같은지 |
| 파일 위치 | JSON 파일 이름과 이미지 파일 이름이 같은지(확장자만 다름), 조합 폴더 이름이 그 이미지의 조합 이름과 같은지 |
| 어노테이션 식별자 | annotation `id`가 겹치지 않는지 |
| 파일명과 촬영 정보 | 1-3에서 값이 여러 개였던 자리와 촬영 정보(`camera_la`, `camera_lo`, `size`, `drug_dir`)를 비교 |

검사마다 맞지 않는 행 수를 세고, 맞지 않는 행이 있는 검사만 그 행을 보여 준다.

In [ ]:
train_name_set = set(train_names)
json_name_set = set(ann_df["file_name"])


def code_number(series, prefix="K-"):
    """'K-003351' 같은 문자열이나 '3351' 같은 숫자 문자열을 정수로 바꾼다. 바꿀 수 없으면 NaN."""
    return pd.to_numeric(series.astype(str).str.removeprefix(prefix), errors="coerce")


ids_per_file = ann_df.groupby("file_name")["image_id"].nunique()
files_per_id = ann_df.groupby("image_id")["file_name"].nunique()
size_per_file = ann_df.groupby("file_name")[["width", "height"]].nunique().max(axis=1)
json_stem = ann_df["json_path"].str.rsplit("/", n=1).str[-1].str.removesuffix(".json")
combo_folder = ann_df["json_path"].str.split("/").str[0]

# 검사 이름 → 맞지 않는 행 (ann_df의 행). 파일 단위 검사는 해당 파일의 행을 모은다.
problems = {
    "이미지 파일이 없는 JSON": ann_df[~ann_df["file_name"].isin(train_name_set)],
    "같은 파일명에 image_id가 여러 개": ann_df[ann_df["file_name"].isin(ids_per_file[ids_per_file > 1].index)],
    "같은 image_id에 파일명이 여러 개": ann_df[ann_df["image_id"].isin(files_per_id[files_per_id > 1].index)],
    "annotations의 image_id ≠ images의 id": ann_df[ann_df["ann_image_id"] != ann_df["image_id"]],
    "같은 이미지인데 width, height가 다름": ann_df[ann_df["file_name"].isin(size_per_file[size_per_file > 1].index)],
    "약 폴더 이름 ≠ category_id": ann_df[code_number(ann_df["folder_code"]) != ann_df["category_id"]],
    "categories의 id ≠ category_id": ann_df[ann_df["cat_id"] != ann_df["category_id"]],
    "dl_idx ≠ category_id": ann_df[code_number(ann_df["dl_idx"], prefix="") != ann_df["category_id"]],
    "drug_N ≠ category_id": ann_df[code_number(ann_df["drug_N"]) != ann_df["category_id"]],
    "categories의 name ≠ dl_name": ann_df[ann_df["cat_name"] != ann_df["dl_name"]],
    "JSON 파일 이름 ≠ 이미지 파일 이름": ann_df[json_stem != ann_df["file_name"].str.rsplit(".", n=1).str[0]],
    "조합 폴더 이름 ≠ 조합 이름": ann_df[combo_folder != ann_df["combo"] + "_json"],
    "annotation id 중복": ann_df[ann_df["ann_id"].duplicated(keep=False)],
}
no_json_images = sorted(train_name_set - json_name_set)  # 이미지는 있는데 JSON이 없음 (ann_df에 행이 없어서 따로 센다)

check_rows = [{"검사": "JSON이 없는 이미지", "맞지 않는 수": len(no_json_images), "단위": "이미지"}]
check_rows += [{"검사": name, "맞지 않는 수": len(bad), "단위": "어노테이션"} for name, bad in problems.items()]
display(pd.DataFrame(check_rows))

if no_json_images:
    print("JSON이 없는 이미지:", no_json_images)
SHOW_COLUMNS = ["json_path", "file_name", "image_id", "ann_image_id", "category_id", "cat_id", "dl_idx", "drug_N", "dl_name", "cat_name", "ann_id"]
for name, bad in problems.items():
    if len(bad):
        print(f"[{name}] {len(bad)}개")
        display(bad[SHOW_COLUMNS])

# 이미지 한 장에 연결된 JSON 수
per_image = ann_df.groupby("file_name").size().value_counts().sort_index()
display(per_image.rename_axis("이미지 한 장의 JSON 수").reset_index(name="이미지 수"))

In [ ]:
# 파일명의 자리와 JSON 촬영 정보 비교 (JSON 하나가 한 행)
parts = ann_df["file_name"].str.rsplit(".", n=1).str[0].str.split("_", expand=True)
compare = [(5, "camera_la"), (6, "camera_lo"), (7, "size")]
compare_rows = []
for col, field in compare:
    same = pd.to_numeric(parts[col], errors="coerce") == pd.to_numeric(ann_df[field], errors="coerce")
    compare_rows.append({"파일명 자리": col, "JSON 항목": field, "같은 JSON 수": int(same.sum()), "다른 JSON 수": int((~same).sum())})
display(pd.DataFrame(compare_rows))

# 3번 자리는 숫자이고 drug_dir은 글자라 직접 비교할 수 없어서, 값끼리 짝을 표로 본다.
display(pd.crosstab(parts[3].rename("파일명 3번 자리"), ann_df["drug_dir"].rename("drug_dir")))

#### 1-7 결과

- 14가지 검사 모두 맞지 않는 수가 0이다.
    - 이미지 232장 모두 JSON이 있고, 이미지 파일이 없는 JSON도 없다.
    - 파일명과 `image_id`가 일대일로 대응하고, `annotations`의 `image_id`는 같은 JSON의 `images`의 `id`와 같다. 같은 이미지의 JSON끼리 `width`, `height`도 같다.
    - 약 폴더 이름, `category_id`, `categories`의 `id`, `dl_idx`, `drug_N`이 모두 같은 번호이고, `categories`의 `name`과 `dl_name`도 같다. 즉 클래스 번호는 알약 코드(`K-` 뒤 숫자)를 정수로 바꾼 값이다.
    - JSON 파일 이름은 이미지 파일 이름과 같고, 조합 폴더 이름은 `<조합 이름>_json`이다. annotation `id`는 겹치지 않는다.
- 이미지 한 장의 JSON 수는 2개가 7장, 3개가 151장, 4개가 74장이다.
- 파일명 5번 자리는 `camera_la`, 6번 자리는 `camera_lo`, 7번 자리는 `size`와 763개 모두 같다. 3번 자리는 0이면 `drug_dir`이 앞면(751개), 1이면 뒷면(12개)으로 짝이 맞는다.

### 1-8. 정리

#### 확인한 것

- 파일: train 이미지 PNG 232장, train 라벨 JSON 763개, test 이미지 PNG 842장이다. 다른 형식의 파일이나 빈 파일은 없다.
- 라벨 구조: JSON 하나에 이미지, 어노테이션, 클래스가 1개씩 있다. 이미지 한 장의 라벨은 그 사진의 알약 수만큼의 JSON에 나뉘어 `<조합 이름>_json/K-<알약 코드>/<이미지 파일명>.json`에 있다. 그래서 이미지 단위로 쓰려면 같은 파일명의 JSON을 모아야 한다.
- 연결: 이미지와 JSON, `image_id`, 클래스 번호(약 폴더 이름, `category_id`, `categories`의 `id`, `dl_idx`, `drug_N`)와 약 이름이 모두 서로 맞는다. 클래스 번호는 알약 코드의 숫자다.
- 파일명 규칙: train 파일명은 `<조합 이름>_0_2_<앞뒷면>_2_<camera_la>_<camera_lo>_<size>.png`다.
    - 조합 이름에는 그 사진에 놓인 알약의 코드가 3개나 4개 들어 있다.
    - 파일명에서 달라지는 촬영 정보는 각도(`camera_la` 70/75/90)와 앞뒷면뿐이다. 뒷면 사진은 4장이다.
- test 파일명은 1~1500 사이의 번호뿐이라, 파일명으로는 조합이나 촬영 정보를 알 수 없다.
- 대회 설명에 없는 항목 `imgfile`이 있다. `dl_idx`는 글자, `category_id`는 정수로 자료형이 다르다.

#### 뒤 섹션에서 볼 것

- JSON이 2개인 이미지가 7장인데, 파일명에는 알약 코드가 2개인 경우가 없다. 파일명에 적힌 알약보다 JSON이 적은 이미지가 있다는 뜻이라, 3. 라벨 품질에서 확인한다.
- 같은 조합 이름의 사진이 여러 장 있다(114종류, 232장). 조합별 사진 수와 사진끼리의 차이는 6. 조합과 촬영 각도에서 본다.
- 값이 빈 문자열인 항목이 있다. 전체에서 얼마나 비어 있는지는 3. 라벨 품질에서 센다.

다음 섹션부터는 `ann_df`를 쓴다.

## 2. 이미지

이미지 자체를 본다. 라벨이 맞는지는 3번에서 보고, 여기서는 사진이 어떻게 생겼는지와 train과 test가 비슷한 조건인지를 확인한다.

| 단계 | 하는 일 | 이유 |
|---|---|---|
| 2-1 | train, test 샘플 이미지 보기 | 사진이 어떻게 생겼고 라벨이 어떻게 붙어 있는지 먼저 눈으로 본다. |
| 2-2 | 이미지 크기와 색 형식 | 입력 크기를 정할 때와 좌표를 다룰 때 기준이 된다. JSON의 `width`, `height`가 실제 크기와 같은지도 본다. |
| 2-3 | 촬영 조건 | 학습 데이터가 얼마나 다양한 조건에서 찍혔는지 본다. |
| 2-4 | train과 test의 밝기와 색 비교 | test에는 촬영 정보가 없어서, 이미지 자체로 조건이 비슷한지 본다. |
| 2-5 | 중복 이미지 | 같은 파일이 train 안에, 또는 train과 test에 함께 있으면 학습과 평가가 왜곡된다. |

같은 조합을 다른 각도로 찍은 비슷한 사진은 파일 내용이 달라서 2-5에서는 잡히지 않는다. 6번에서 다룬다.

### 2-1. 샘플 이미지

- train과 test에서 6장씩 무작위로 뽑는다. 셀을 다시 실행해도 같은 사진이 나오도록, 셀 안에서 0-4의 `SEED`로 새로 뽑는다.
- train에는 JSON의 bbox를 초록색 사각형으로 그리고, 왼쪽 위에 클래스 번호(`category_id`)를 적는다. 제목에는 조합 이름과 촬영 각도(`camera_la`)를 적는다.
- test는 라벨이 없어서 이미지만 보여 준다.

In [ ]:
from matplotlib.patches import Rectangle

N_SAMPLES = 6
sample_rng = np.random.default_rng(SEED)  # 셀을 다시 실행해도 같은 사진이 뽑히도록 셀 안에서 새로 만든다
train_samples = [train_img_paths[i] for i in sorted(sample_rng.choice(len(train_img_paths), N_SAMPLES, replace=False))]
test_samples = [test_img_paths[i] for i in sorted(sample_rng.choice(len(test_img_paths), N_SAMPLES, replace=False))]


def draw_boxes(ax, rows):
    """ann_df의 행들(이미지 하나의 어노테이션)을 bbox 사각형과 클래스 번호로 그린다."""
    for r in rows.itertuples():
        ax.add_patch(Rectangle((r.bbox_x, r.bbox_y), r.bbox_w, r.bbox_h, fill=False, edgecolor="lime", linewidth=2))
        ax.text(r.bbox_x, r.bbox_y - 8, str(r.category_id), fontsize=8, color="white", va="bottom",
                bbox=dict(facecolor="green", alpha=0.85, pad=1.5, linewidth=0))


fig, axes = plt.subplots(2, 3, figsize=(12, 11), layout="constrained")  # 전체 제목과 사진 제목이 겹치지 않게
for ax, p in zip(axes.flat, train_samples):
    rows = ann_df[ann_df["file_name"] == p.name]
    ax.imshow(Image.open(p))
    draw_boxes(ax, rows)
    ax.set_title(f"{combo_name(p.name)}\n각도 {rows['camera_la'].iloc[0]}°, bbox {len(rows)}개", fontsize=9)
    ax.axis("off")
fig.suptitle("train 샘플 (초록 사각형: bbox, 숫자: category_id)")
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(12, 11), layout="constrained")  # 전체 제목과 사진 제목이 겹치지 않게
for ax, p in zip(axes.flat, test_samples):
    ax.imshow(Image.open(p))
    ax.set_title(p.name, fontsize=9)
    ax.axis("off")
fig.suptitle("test 샘플")
plt.show()

#### 2-1 결과 (그림에서 보이는 것)

- train: 연한 회색(사진에서는 푸른빛이 도는 회색) 바닥 위에 알약이 3~4개씩 서로 떨어져 놓여 있다. 6장 모두 알약 하나에 bbox가 하나씩 있고, bbox는 알약 가장자리에 맞춰 그려져 있다.
- 알약 표면에 글자나 숫자(각인)가 있다. 모양과 색이 비슷한 흰색 원형 알약도 각인이 서로 다르다(여섯 번째 사진의 3544와 4543).
- 일부 사진은 모서리에 밝은 원형 테두리가 보인다(train 첫 번째, 두 번째, 네 번째 사진).
- test: train과 같은 바닥과 조명으로 보이고, 알약이 3~4개씩 놓여 있다. 694.png처럼 밝은 원형 테두리가 보이는 사진도 있다.

샘플 12장을 보고 적은 것이라 전체에 대한 결론은 아니다. 라벨이 모든 사진에서 맞는지는 3번에서 확인한다.

### 2-2. 이미지 크기와 색 형식

- train, test 이미지를 모두 열어 실제 픽셀 크기(너비 × 높이)와 색 형식을 센다. 이미지를 열 때 파일의 머리 부분만 읽기 때문에 빠르다.
    - 색 형식은 PIL의 `mode`다. `RGB`는 색 3채널, `RGBA`는 투명도까지 4채널, `L`은 흑백이다. 형식이 섞여 있으면 학습 전에 하나로 맞춰야 한다.
- train은 JSON의 `width`, `height`가 실제 크기와 같은지도 비교한다. bbox 좌표가 이 크기를 기준으로 적혀 있기 때문이다.

In [ ]:
def image_info(path):
    """이미지의 (너비, 높이, 색 형식)을 돌려준다. 파일 머리 부분만 읽어서 빠르다."""
    with Image.open(path) as im:
        return im.size[0], im.size[1], im.mode


train_info = pd.DataFrame([(p.name, *image_info(p)) for p in train_img_paths], columns=["file_name", "real_w", "real_h", "mode"])
test_info = pd.DataFrame([(p.name, *image_info(p)) for p in test_img_paths], columns=["file_name", "real_w", "real_h", "mode"])

size_table = pd.concat([train_info.assign(구분="train"), test_info.assign(구분="test")])
display(size_table.groupby(["구분", "real_w", "real_h", "mode"], sort=False).size()
        .reset_index(name="이미지 수").rename(columns={"real_w": "너비", "real_h": "높이", "mode": "색 형식"}))

# train: JSON의 width, height와 실제 크기 비교 (1-7에서 같은 이미지의 JSON끼리는 width, height가 같았다)
json_size = ann_df.drop_duplicates("file_name")[["file_name", "width", "height"]]
size_check = train_info.merge(json_size, on="file_name")
size_diff = size_check[(size_check["real_w"] != size_check["width"]) | (size_check["real_h"] != size_check["height"])]
print(f"JSON의 width, height가 실제 크기와 다른 train 이미지: {len(size_diff)}장 / {len(size_check)}장")
if len(size_diff):
    display(size_diff)

#### 2-2 결과

- train 232장과 test 842장 모두 976 × 1280(너비 × 높이), `RGB`다. 크기와 색 형식이 하나로 같다.
- train의 JSON `width`, `height`는 232장 모두 실제 크기와 같다.

### 2-3. 촬영 조건

- 촬영 정보는 이미지에 대한 값이라, 이미지 한 장당 한 줄로 센다. JSON 단위로 세면 알약이 많은 사진이 여러 번 세어진다.
- 그 전에, 같은 이미지의 JSON끼리 촬영 정보가 모두 같은지 확인한다. 같아야 이미지 한 장의 값으로 볼 수 있다.
- 항목: `drug_S`(알약 상태), `back_color`(배경), `light_color`(조명), `drug_dir`(앞뒷면), `camera_la`(카메라 위도), `camera_lo`(카메라 경도), `size`(이미지 사이즈)

In [ ]:
SHOOT_FIELDS = ["drug_S", "back_color", "light_color", "drug_dir", "camera_la", "camera_lo", "size"]

# 같은 이미지의 JSON끼리 촬영 정보가 다른 항목 수
varying = ann_df.groupby("file_name")[SHOOT_FIELDS].nunique()
varying_images = varying[(varying > 1).any(axis=1)]
print(f"같은 이미지의 JSON끼리 촬영 정보가 다른 이미지: {len(varying_images)}장")
if len(varying_images):
    display(varying_images)

image_df = ann_df.drop_duplicates("file_name")[["file_name", "combo", *SHOOT_FIELDS]].reset_index(drop=True)
shoot_rows = []
for field in SHOOT_FIELDS:
    counts = image_df[field].value_counts()
    shoot_rows.append({
        "항목": field,
        "값 종류 수": len(counts),
        "값 분포 (값: 이미지 수)": ", ".join(f"{v}: {n}" for v, n in counts.items()),
    })
print(f"이미지 {len(image_df)}장 기준")
display(pd.DataFrame(shoot_rows))

#### 2-3 결과

- 같은 이미지의 JSON끼리 촬영 정보가 다른 경우는 없다. 그래서 이미지 한 장의 값으로 셌다.
- 이미지 232장 기준

| 항목 | 값 |
|---|---|
| `drug_S`(알약 상태) | 정상알약 232장 |
| `back_color`(배경) | 연회색 배경 232장 |
| `light_color`(조명) | 주백색 232장 |
| `drug_dir`(앞뒷면) | 앞면 228장, 뒷면 4장 |
| `camera_la`(카메라 위도) | 90이 85장, 75가 75장, 70이 72장 |
| `camera_lo`(카메라 경도) | 0 232장 |
| `size` | 200 232장 |

- 배경, 조명, 알약 상태, 카메라 경도, 사이즈는 모두 한 가지 값이다. 달라지는 것은 카메라 위도(세 값이 비슷한 비율)와 앞뒷면(뒷면은 4장)뿐이다.

### 2-4. train과 test의 밝기와 색 비교

- test에는 촬영 정보가 없어서, 이미지의 픽셀 값으로 train과 비슷한 조건인지 본다.
- 이미지마다 R, G, B 채널의 평균값(0~255)을 구하고, 세 채널의 평균을 밝기로 본다. 배경과 조명이 다르면 이 값의 분포가 달라진다.
- 계산을 빠르게 하려고 이미지를 가로세로 1/4로 줄여서 평균을 낸다. 줄일 때 주변 픽셀의 평균으로 줄이기 때문에, 평균값은 원본과 거의 같다. 이미지 1,000여 장을 열어서 시간이 조금 걸린다.
- train과 test의 이미지 수가 달라서, 그래프의 세로축은 개수 대신 비율(밀도)로 그린다.

In [ ]:
def channel_means(path, scale=4):
    """이미지를 1/scale로 줄인 뒤 R, G, B 채널별 평균(0~255)을 돌려준다."""
    with Image.open(path) as im:
        im = im.convert("RGB")
        small = im.reduce(scale)  # scale × scale 칸의 평균으로 줄인다
    return np.asarray(small, dtype=np.float64).mean(axis=(0, 1))


color_df = pd.concat([
    pd.DataFrame([channel_means(p) for p in train_img_paths], columns=["R", "G", "B"]).assign(구분="train"),
    pd.DataFrame([channel_means(p) for p in test_img_paths], columns=["R", "G", "B"]).assign(구분="test"),
], ignore_index=True)
color_df["밝기"] = color_df[["R", "G", "B"]].mean(axis=1)

display(color_df.groupby("구분", sort=False)[["R", "G", "B", "밝기"]].agg(["mean", "std", "min", "max"]).round(1))

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), layout="constrained")
for ax, col in zip(axes, ["R", "G", "B", "밝기"]):
    bins = np.linspace(color_df[col].min(), color_df[col].max(), 40)
    for name, color in [("train", "tab:blue"), ("test", "tab:orange")]:
        ax.hist(color_df.loc[color_df["구분"] == name, col], bins=bins, density=True, alpha=0.5, color=color, label=name)
    ax.set_title(f"이미지별 {col} 평균")
    ax.set_xlabel("평균값 (0~255)")
axes[0].set_ylabel("밀도")
axes[0].legend()
plt.show()

#### 2-4 결과

| 값 | train 평균 (표준편차) | test 평균 (표준편차) |
|---|---|---|
| R | 115.7 (13.1) | 111.6 (16.6) |
| G | 129.7 (15.6) | 121.1 (18.6) |
| B | 148.7 (10.0) | 144.7 (11.0) |
| 밝기 | 131.4 (12.2) | 125.8 (14.6) |

- train과 test 모두 세 채널 중 B의 평균이 가장 높다. 2-1에서 바닥이 푸른빛 회색으로 보인 것과 같다.
- test가 train보다 조금 어둡다. 밝기 평균이 5.6 낮고, 채널 중에서는 G 평균이 8.6 낮아 차이가 가장 크다.
- test가 이미지마다 값의 차이가 더 크다. 네 값 모두 표준편차가 train보다 크고, 밝기 최솟값도 test가 더 낮다(96.5, train은 103.6).
- 분포 그래프를 보면 두 분포의 범위는 대부분 겹친다. test는 G와 밝기에서 train보다 왼쪽(어두운 쪽)에 더 많이 몰려 있다.

### 2-5. 중복 이미지

- 파일 내용의 MD5 해시를 구해서 같은 값끼리 묶는다. 해시는 파일 내용으로 만든 지문 같은 값이라, 내용이 1바이트라도 다르면 값이 달라진다. 그래서 해시가 같으면 완전히 같은 파일이다.
- train 안, test 안, train과 test 사이에서 해시가 같은 파일 수를 센다. 같은 파일이 있으면 그 목록을 보여 준다.
- 모든 이미지 파일(약 1.8GB)을 읽어서 시간이 조금 걸린다.

In [ ]:
import hashlib


def file_md5(path):
    """파일 내용의 MD5 해시(32자리 16진수 글자)를 돌려준다."""
    return hashlib.md5(path.read_bytes()).hexdigest()


hash_df = pd.DataFrame(
    [("train", p.name, file_md5(p)) for p in train_img_paths]
    + [("test", p.name, file_md5(p)) for p in test_img_paths],
    columns=["구분", "file_name", "md5"],
)

same_hash = hash_df[hash_df["md5"].duplicated(keep=False)]  # 해시가 다른 파일과 같은 행
groups = same_hash.groupby("md5")["구분"].agg(lambda s: "+".join(sorted(set(s))))
dup_rows = [
    {"비교": "train 안", "같은 파일 묶음 수": int((groups == "train").sum())},
    {"비교": "test 안", "같은 파일 묶음 수": int((groups == "test").sum())},
    {"비교": "train과 test 사이", "같은 파일 묶음 수": int((groups == "test+train").sum())},
]
display(pd.DataFrame(dup_rows))
if len(same_hash):
    display(same_hash.sort_values(["md5", "구분", "file_name"]))

#### 2-5 결과

- train 안, test 안, train과 test 사이 모두 완전히 같은 파일은 없다.

### 2-6. 정리

#### 확인한 것

- 크기와 형식: train과 test 이미지 모두 976 × 1280, `RGB`로 같다. JSON의 크기 정보도 실제와 같다.
- 촬영 조건(train): 배경, 조명, 알약 상태, 카메라 경도, 사이즈는 한 가지뿐이다. 달라지는 것은 카메라 위도(70, 75, 90이 비슷한 비율)와 앞뒷면(뒷면 4장)이다. train은 다양한 촬영 조건을 담고 있지 않다.
- train과 test: 사진으로는 같은 바닥과 조명에서 찍은 것으로 보인다. 픽셀 평균은 test가 조금 더 어둡고(특히 G 채널), 이미지마다 차이가 더 크지만, 값의 범위는 대부분 겹친다.
- 중복: 완전히 같은 파일은 train 안, test 안, train과 test 사이 모두 없다.

#### 뒤 섹션에서 볼 것

- 파일 내용이 다른 비슷한 사진(같은 조합을 다른 각도로 찍은 사진)은 6. 조합과 촬영 각도에서 본다.

#### 이 EDA에서 확인하지 않은 것

- test에는 촬영 정보가 없어서, test의 각도나 앞뒷면 비율은 이미지로만 짐작할 수 있다. 이 EDA에서는 확인하지 않았다.

## 3. 라벨 품질

대회 설명에 "제공된 데이터에는 크고 작은 오류가 포함되어 있다"고 되어 있다. 라벨에 어떤 오류가 있는지 종류별로 찾고, 고친다면 어떻게 고칠 수 있는지 방안을 정리한다. 이 노트북에서는 오류를 고치지 않는다.

| 단계 | 찾는 오류 | 방법 |
|---|---|---|
| 3-1 | 라벨 누락 | 이미지마다 파일명의 알약 코드와 JSON의 클래스를 비교 |
| 3-2 | 이미지 밖 bbox | bbox 좌표가 이미지 범위를 벗어나는지, 너비와 높이가 0보다 큰지 확인 |
| 3-3 | 같은 이미지 안 bbox 겹침 | 서로 다른 클래스의 bbox가 겹치는 쌍을 찾음 |
| 3-4 | 클래스별 bbox 영역 모아 보기 | 클래스마다 bbox 영역을 잘라 모아서, 다른 알약이나 빈 바닥이 잘린 칸을 찾음 |
| 3-5 | 3-4에서 다르게 보인 칸 | 그 칸의 원래 사진을 보고 무엇이 잘못됐는지 확인 |
| 3-6 | area 불일치 | `area`와 bbox의 너비 × 높이를 비교 |
| 3-7 | 결측치 | JSON 항목마다 비어 있는 값을 셈 |
| 3-8 | 오류 목록과 고치는 방안 | 찾은 오류를 한 표로 모으고 고치는 방안을 적음 |

bbox는 `[x, y, w, h]` 형식이다. `x`, `y`는 왼쪽 위 꼭짓점, `w`, `h`는 너비와 높이이고, 단위는 픽셀이다.

### 3-1. 라벨 누락

- 1-3에서 조합 이름에 그 사진에 놓인 알약의 코드가 들어 있는 것을 봤고, 1-7에서 클래스 번호가 알약 코드의 숫자인 것을 확인했다. 그래서 이미지마다 파일명의 알약 코드와 JSON의 `category_id`를 비교하면, 사진에 있는데 라벨이 없는 알약을 찾을 수 있다.
- 이미지마다 비교한다.
    - 파일명에는 있는데 JSON에는 없는 코드: 라벨 누락 후보
    - JSON에는 있는데 파일명에는 없는 코드: 사진에 없는 알약의 라벨이거나 클래스 번호 오류 후보
- 후보 이미지에는 기존 bbox를 모두 그려서, bbox가 없는 알약이 실제로 있는지 눈으로 확인한다.
- 이미지는 그리는 시간을 줄이려고 작게 줄여서 그린다. bbox는 원본 좌표 그대로 그린다.

In [ ]:
img_labels = pd.DataFrame({"file_name": train_names})
img_labels["codes"] = img_labels["file_name"].apply(pill_codes)  # 1-3의 함수: 파일명의 알약 코드
classes_by_name = ann_df.groupby("file_name")["category_id"].apply(list)
img_labels["classes"] = [classes_by_name.get(name, []) for name in img_labels["file_name"]]
img_labels["n_codes"] = img_labels["codes"].apply(len)
img_labels["n_bbox"] = img_labels["classes"].apply(len)
img_labels["missing"] = [sorted(set(c) - set(a)) for c, a in zip(img_labels["codes"], img_labels["classes"])]  # 파일명에만 있음
img_labels["extra"] = [sorted(set(a) - set(c)) for c, a in zip(img_labels["codes"], img_labels["classes"])]    # JSON에만 있음

summary = [
    {"비교": "파일명의 코드와 JSON의 클래스가 같음", "이미지 수": int(((img_labels["missing"].apply(len) == 0) & (img_labels["extra"].apply(len) == 0)).sum())},
    {"비교": "파일명에만 있는 코드가 있음 (라벨 누락 후보)", "이미지 수": int((img_labels["missing"].apply(len) > 0).sum())},
    {"비교": "JSON에만 있는 클래스가 있음", "이미지 수": int((img_labels["extra"].apply(len) > 0).sum())},
]
display(pd.DataFrame(summary))

missing_df = img_labels[(img_labels["missing"].apply(len) > 0) | (img_labels["extra"].apply(len) > 0)]
if len(missing_df):
    display(missing_df[["file_name", "n_codes", "n_bbox", "missing", "extra"]])

In [ ]:
path_by_name = {p.name: p for p in train_img_paths}


def show_thumbs(paths, captions, boxes, title="", ncols=4, thumb=400):
    """이미지를 줄여서 격자로 보여 주고, 원본 좌표 기준의 bbox를 그린다.

    boxes: 이미지마다 [(x, y, w, h, 글, 색), ...]. 좌표가 같은 bbox가 여러 개면 글을 아래로 차례로 적어 겹치지 않게 한다.
    """
    nrows = (len(paths) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.8 * ncols, 4.1 * nrows), layout="constrained", squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, path, caption, image_boxes in zip(axes.flat, paths, captions, boxes):
        with Image.open(path) as im:
            w, h = im.size
            im.thumbnail((thumb, thumb))  # 비율을 유지한 채 긴 변이 thumb 픽셀이 되도록 줄임
            ax.imshow(im, extent=(0, w, h, 0))  # 줄인 이미지를 원본 크기의 좌표 위에 펼쳐 그림
        stacked = Counter()
        for x, y, bw, bh, label, color in image_boxes:
            ax.add_patch(Rectangle((x, y), bw, bh, fill=False, edgecolor=color, linewidth=1.5))
            ax.text(x + 8, y + 8 + 60 * stacked[(x, y, bw, bh)], label, fontsize=7, color="white", va="top", clip_on=True,
                    bbox=dict(facecolor=color, edgecolor="none", pad=1))
            stacked[(x, y, bw, bh)] += 1
        ax.set_title(caption, fontsize=7)
    fig.suptitle(title, fontsize=11)
    plt.show()


def boxes_for(file_name, color="#e5484d"):
    """이미지 하나의 bbox를 show_thumbs에 넘길 형태로 만든다. 글은 category_id."""
    rows = ann_df[ann_df["file_name"] == file_name]
    return [(r.bbox_x, r.bbox_y, r.bbox_w, r.bbox_h, str(r.category_id), color) for r in rows.itertuples()]


if len(missing_df):
    show_thumbs(
        paths=[path_by_name[n] for n in missing_df["file_name"]],
        captions=[f"{r.file_name.split('_')[0]}\n각도 {r.file_name.split('_')[5]}°, 코드 {r.n_codes}개 / bbox {r.n_bbox}개\n없음: {r.missing}"
                  for r in missing_df.itertuples()],
        boxes=[boxes_for(n) for n in missing_df["file_name"]],
        title="라벨 누락 후보 (빨간 사각형: JSON의 bbox, 글: category_id)",
    )

후보가 아닌 이미지(파일명의 코드와 JSON의 클래스가 같은 이미지)에서도 6장을 뽑아 같은 방식으로 본다. 파일명과 개수가 맞아도 bbox가 엉뚱한 곳에 있을 수 있어서, 후보가 아닌 사진도 몇 장은 눈으로 확인한다. bbox 위치 오류는 3-3~3-5에서 따로 찾는다.

In [ ]:
ok_names = img_labels.loc[~img_labels.index.isin(missing_df.index), "file_name"].tolist()
picked = sorted(np.random.default_rng(SEED).choice(len(ok_names), 6, replace=False))
picked_names = [ok_names[i] for i in picked]
show_thumbs(
    paths=[path_by_name[n] for n in picked_names],
    captions=[f"{n.split('_')[0]}\n각도 {n.split('_')[5]}°" for n in picked_names],
    boxes=[boxes_for(n) for n in picked_names],
    title="후보가 아닌 이미지 6장 (무작위)",
    ncols=3,
)

#### 3-1 결과

- 224장은 파일명의 코드와 JSON의 클래스가 같다. 8장은 파일명에만 있는 코드가 하나씩 있다. JSON에만 있는 클래스는 없다.
- 8장 중 7장은 코드 3개에 bbox 2개, 1장은 코드 4개에 bbox 3개다. 1-8에서 본 "JSON이 2개인 이미지 7장"이 이 7장이다.
- 빠진 코드는 3351이 4장, 20014, 32310, 33880, 4543이 1장씩이다.

그림에서 보이는 것

- 8장 모두 bbox가 없는 알약이 하나씩 있다.
- 3351이 빠진 4장은 bbox가 없는 알약이 HT 각인의 주황색 원형 알약이다. 다른 사진(2-1 샘플, 후보가 아닌 사진)에서 3351 bbox가 붙은 알약과 모양이 같다.
- 4543이 빠진 사진은 Airtal 각인의 흰색 원형 알약에 bbox가 없다. 2-1 여섯 번째 사진에서 4543 bbox가 붙은 알약과 모양이 같다.
- 20014, 32310, 33880이 빠진 사진은 각각 왼쪽의 큰 갈색 원형 알약(십자 분할선), 오른쪽 아래의 노란색 타원형 알약, 왼쪽 아래의 JIC 각인 노란색 알약에 bbox가 없다.
- 후보가 아닌 6장은 모든 알약에 bbox가 있다.

8장 모두 파일명에 있는 알약 하나의 라벨이 빠진 라벨 누락으로 보인다. 고치는 방안은 3-8에서 정리한다.

### 3-2. 이미지 밖 bbox

- bbox마다 오른쪽 아래 꼭짓점(`x + w`, `y + h`)을 구해서, 이미지 범위(`0 ~ width`, `0 ~ height`) 안에 있는지 본다.
- 너비나 높이가 0 이하인 bbox도 센다.
- 좌표와 크기의 최솟값, 최댓값을 표로 보여 준다. 이미지 크기(976 × 1280)와 비교하면 범위를 벗어난 값이 드러난다.
- 이미지 밖 bbox가 있으면, 같은 조합의 다른 각도 사진과 함께 그려 본다. 이미지 밖 bbox는 사진에 그릴 수 없어서 값만 제목에 적는다.
- 좌표의 숫자 한 자리가 잘못 들어간 오타라면, 숫자를 한 자리씩 빼 본 값 중 이미지 안에 들어가는 값이 후보가 된다. 후보마다 bbox를 그려서 알약을 감싸는 값을 찾는다.

In [ ]:
box_df = ann_df[["file_name", "category_id", "width", "height", "bbox_x", "bbox_y", "bbox_w", "bbox_h"]].copy()
box_df["x2"] = box_df["bbox_x"] + box_df["bbox_w"]  # 오른쪽 아래 꼭짓점
box_df["y2"] = box_df["bbox_y"] + box_df["bbox_h"]

non_positive = (box_df["bbox_w"] <= 0) | (box_df["bbox_h"] <= 0)
out_x = (box_df["bbox_x"] < 0) | (box_df["x2"] > box_df["width"])
out_y = (box_df["bbox_y"] < 0) | (box_df["y2"] > box_df["height"])
outside = out_x | out_y

print(f"검사한 bbox: {len(box_df)}개")
print(f"너비나 높이가 0 이하: {non_positive.sum()}개")
print(f"이미지 밖 (가로 방향 {out_x.sum()}개, 세로 방향 {out_y.sum()}개): {outside.sum()}개")
display(box_df[["bbox_x", "bbox_y", "x2", "y2", "bbox_w", "bbox_h"]].agg(["min", "max"]))

out_df = box_df[outside | non_positive]
if len(out_df):
    display(out_df)

In [ ]:
def inside_boxes(file_name):
    """이미지 하나에서 이미지 안에 있는 bbox만 show_thumbs에 넘길 형태로 만든다."""
    rows = box_df[(box_df["file_name"] == file_name) & ~outside]
    return [(r.bbox_x, r.bbox_y, r.bbox_w, r.bbox_h, str(r.category_id), "#e5484d") for r in rows.itertuples()]


for r in box_df[outside].itertuples():
    same_combo = sorted(n for n in train_names if combo_name(n) == combo_name(r.file_name))
    show_thumbs(
        paths=[path_by_name[n] for n in same_combo],
        captions=[f"각도 {n.split('_')[5]}°" + (f"\n이미지 밖: {r.category_id} [{r.bbox_x}, {r.bbox_y}, {r.bbox_w}, {r.bbox_h}]" if n == r.file_name else "")
                  for n in same_combo],
        boxes=[inside_boxes(n) for n in same_combo],
        title=f"이미지 밖 bbox가 있는 사진과 같은 조합의 사진: {combo_name(r.file_name)}",
        ncols=len(same_combo),
    )

    # 범위를 벗어난 방향의 좌표에서 숫자를 한 자리씩 뺀 값 중, bbox가 이미지 안에 들어가는 값을 후보로 그린다.
    axis, value, size, limit = ("x", r.bbox_x, r.bbox_w, r.width) if out_x[r.Index] else ("y", r.bbox_y, r.bbox_h, r.height)
    digits = str(int(value))
    candidates = sorted({int(digits[:i] + digits[i + 1:]) for i in range(len(digits)) if len(digits) > 1})
    candidates = [c for c in candidates if 0 <= c and c + size <= limit]
    print(f"{r.file_name} / {r.category_id}: {axis} = {value}, 숫자 한 자리를 뺀 후보 {candidates}")
    if candidates:
        cand_boxes = [(c, r.bbox_y, r.bbox_w, r.bbox_h) if axis == "x" else (r.bbox_x, c, r.bbox_w, r.bbox_h) for c in candidates]
        show_thumbs(
            paths=[path_by_name[r.file_name]] * len(candidates),
            captions=[f"후보 {axis} = {c}" for c in candidates],
            boxes=[inside_boxes(r.file_name) + [(*b, f"{r.category_id} ({axis}={c})", "#2a78d6")] for b, c in zip(cand_boxes, candidates)],
            title=f"{r.category_id}의 {axis} 후보 (파란 사각형: 후보 위치, 빨간 사각형: 다른 bbox)",
            ncols=len(candidates),
        )

#### 3-2 결과

- 너비나 높이가 0 이하인 bbox는 없다.
- 이미지 밖 bbox는 1개다. `K-003351-016262-018357_0_2_0_2_75_000_200.png`의 18357 bbox `[6567, 625, 311, 315]`로, x가 6567이라 이미지 너비(976)를 넘는다. 좌표 최댓값 표의 `bbox_x` 6567과 `x2` 6878이 이 bbox다.
- 이 조합의 사진은 2장(75°, 90°)이다.

그림에서 보이는 것

- 75° 사진에서는 노란색 타원형 알약에 bbox가 보이지 않는다. 90° 사진에서는 같은 알약에 18357 bbox가 붙어 있다.
- x에서 숫자를 한 자리씩 뺀 후보는 567, 656, 657이다. 567은 노란색 알약을 감싸고, 656과 657은 알약의 왼쪽 부분이 bbox 밖으로 나간다.

x 앞에 숫자 6이 하나 더 들어간 오타로 보이며, x = 567이 맞는 값으로 보인다. y, w, h(625, 311, 315)는 그대로 둔 후보다.

### 3-3. 같은 이미지 안 bbox 겹침

- 2-1과 3-1에서 본 사진처럼 알약은 서로 떨어져 놓여 있다. 그래서 서로 다른 알약의 bbox가 겹치면 bbox 위치가 잘못됐을 가능성이 있다.
- 같은 이미지의 bbox를 두 개씩 짝지어 IoU를 계산한다. IoU는 두 bbox가 겹친 넓이를 두 bbox를 합친 넓이로 나눈 값이다. 1이면 좌표가 완전히 같고, 0이면 전혀 겹치지 않는다.
- 이미지 밖 bbox(3-2)는 사진에 그릴 수 없어서 짝에서 뺀다.
- 겹치는 쌍이 있으면 사진 전체와 두 bbox 부분을 확대한 그림을 나란히 보여 준다. 좌표가 완전히 같은 bbox는 사각형이 하나로 보여서, 클래스 번호를 아래로 차례로 적는다.

In [ ]:
def iou(a, b):
    """두 bbox [x, y, w, h]의 IoU(겹친 넓이 / 두 bbox를 합친 넓이)를 계산한다."""
    ax_, ay, aw, ah = a
    bx, by, bw, bh = b
    inter_w = max(0, min(ax_ + aw, bx + bw) - max(ax_, bx))  # 겹친 영역의 너비. 겹치지 않으면 0
    inter_h = max(0, min(ay + ah, by + bh) - max(ay, by))
    inter = inter_w * inter_h
    union = aw * ah + bw * bh - inter
    return inter / union if union > 0 else 0.0


pair_rows = []
for file_name, rows in box_df[~outside].groupby("file_name"):
    boxes = list(rows.itertuples())
    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            a, b = boxes[i], boxes[j]
            bbox_a = [a.bbox_x, a.bbox_y, a.bbox_w, a.bbox_h]
            bbox_b = [b.bbox_x, b.bbox_y, b.bbox_w, b.bbox_h]
            pair_rows.append({"file_name": file_name, "class_a": a.category_id, "class_b": b.category_id,
                              "iou": round(iou(bbox_a, bbox_b), 4), "bbox_a": bbox_a, "bbox_b": bbox_b})
pair_df = pd.DataFrame(pair_rows)

same = pair_df["iou"] == 1
partial = (pair_df["iou"] > 0) & ~same
print(f"검사한 bbox 쌍: {len(pair_df)}개 (이미지 {pair_df['file_name'].nunique()}장)")
print(f"좌표가 완전히 같은 쌍 (IoU = 1): {same.sum()}개")
print(f"조금이라도 겹치는 쌍 (0 < IoU < 1): {partial.sum()}개")
print(f"겹치지 않는 쌍 (IoU = 0): {(pair_df['iou'] == 0).sum()}개")

overlap_df = pair_df[pair_df["iou"] > 0].sort_values("iou", ascending=False)
if len(overlap_df):
    display(overlap_df)

In [ ]:
MARGIN = 60  # 확대 그림에서 두 bbox 바깥으로 더 보여 줄 여백(픽셀)

for r in overlap_df.itertuples():
    with Image.open(path_by_name[r.file_name]) as im:
        img = im.convert("RGB")  # 파일을 닫은 뒤에도 쓸 수 있도록 픽셀을 읽어 둔다

    fig, axes = plt.subplots(1, 2, figsize=(10, 6), layout="constrained")
    for ax in axes:
        ax.imshow(img)
        stacked = Counter()
        for x, y, w, h, label, color in inside_boxes(r.file_name):
            ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=1.5))
            ax.text(x + 8, y + 8 + 40 * stacked[(x, y, w, h)], label, fontsize=8, color="white", va="top", clip_on=True,
                    bbox=dict(facecolor=color, edgecolor="none", pad=1))
            stacked[(x, y, w, h)] += 1
        ax.axis("off")

    # 오른쪽: 두 bbox를 모두 덮는 영역에 여백을 더해 확대한다. 이미지 밖으로는 넘지 않게 한다.
    (xa, ya, wa, ha), (xb, yb, wb, hb) = r.bbox_a, r.bbox_b
    axes[1].set_xlim(max(0, min(xa, xb) - MARGIN), min(img.width, max(xa + wa, xb + wb) + MARGIN))
    axes[1].set_ylim(min(img.height, max(ya + ha, yb + hb) + MARGIN), max(0, min(ya, yb) - MARGIN))  # 이미지 좌표는 아래로 갈수록 y가 커서 아래쪽 값을 먼저 넣는다

    axes[0].set_title("사진 전체", fontsize=9)
    axes[1].set_title(f"확대: {r.class_a}와 {r.class_b}, IoU {r.iou}", fontsize=9)
    fig.suptitle(r.file_name, fontsize=10)
    plt.show()

#### 3-3 결과

- 같은 이미지의 bbox 쌍 902개 중, 좌표가 완전히 같은 쌍이 3개, 조금 겹치는 쌍이 2개, 겹치지 않는 쌍이 897개다.

그림에서 보이는 것

- 좌표가 같은 3쌍은 모두 알약 하나에 두 클래스의 bbox가 같은 좌표로 붙어 있고, 같은 사진에 bbox가 없는 알약이 하나 있다.

| 사진 (모두 70°) | 같은 좌표의 두 클래스 | bbox가 붙은 알약 | bbox가 없는 알약 | 잘못 놓인 것으로 보이는 클래스 |
|---|---|---|---|---|
| `K-001900-016548-019607-033009` | 16548, 33009 | S300 각인 갈색 알약 | 노란색 원형 알약 | 16548 (2-1 여섯 번째 사진에서 노란색 원형 알약에 붙어 있음) |
| `K-003351-020238-031863` | 3351, 20238 | HT 각인 주황색 알약 | 75 각인 분홍색 원형 알약 | 20238 (2-1 다섯 번째 사진에서 75 각인 알약에 붙어 있음) |
| `K-003351-029667-031863` | 3351, 31863 | GIL 각인 흰색 알약 | HT 각인 주황색 알약 | 3351 (3-1에서 HT 각인 알약이 3351로 보임) |

- 조금 겹치는 2쌍은 같은 조합(`K-003351-033880-035206`)의 75°, 90° 사진에 있는 33880과 35206이다(IoU 0.0212, 0.0161). 두 알약이 가까이 놓여 있어서 bbox의 가장자리만 겹치고, 각 bbox는 자기 알약을 감싸고 있다.

좌표가 같은 3쌍은 한 클래스의 bbox가 다른 알약 위에 잘못 놓인 위치 오류로 보인다. 이 3장은 클래스 번호가 파일명과 모두 맞아서 3-1에서는 후보로 잡히지 않았다. 숫자 비교로는 찾을 수 없고 bbox 위치를 봐야 찾을 수 있는 오류다. 조금 겹치는 2쌍은 오류가 아닌 것으로 보인다.

### 3-4. 클래스별 bbox 영역 모아 보기

- 3-1은 파일명과 클래스 번호를, 3-2와 3-3은 좌표 숫자를 비교했다. 그래서 클래스 번호와 좌표 형식은 맞는데 bbox가 다른 알약이나 빈 곳에 그려진 경우는 일부만 찾을 수 있었다(3-3의 좌표가 같은 3쌍).
- 같은 `category_id`의 bbox 영역을 잘라 한곳에 모으면, 모두 같은 모양의 알약이어야 한다. 모양이 다른 알약이나 빈 바닥이 잘린 칸이 있으면 그 bbox가 잘못됐을 가능성이 있다.
- 잘라 낸 영역을 보면 bbox가 알약을 제대로 감싸는지(알약이 잘리거나, bbox가 알약보다 너무 큰지)도 함께 볼 수 있다.

방법

- 3-2의 이미지 밖 bbox 1개는 잘라 낼 수 없어서 뺀다.
- 클래스마다 잘라 낸 영역을 100 × 100 칸 안에 들어가게 줄여서, 한 줄에 12칸씩 붙인다. 칸마다 왼쪽 위의 빨간 숫자는 그 클래스 안에서의 순서다. 다르게 보이는 칸을 찾으면 클래스 번호와 이 숫자로 원래 사진을 찾는다(3-5).
- 그림 하나에 8개 클래스씩 넣는다. 묶음 제목에는 클래스 번호, 약 이름, bbox 수(`n`)를 적는다.

In [ ]:
from PIL import ImageDraw, ImageFont

crop_df = box_df[~outside]  # 3-2의 이미지 밖 bbox 제외. 인덱스는 ann_df와 같다
crop_count = crop_df["category_id"].value_counts()
class_names = ann_df.groupby("category_id")["dl_name"].first()

print(f"잘라 낼 bbox: {len(crop_df)}개 / 클래스: {crop_count.size}개")
print(f"클래스별 bbox 수: 최소 {crop_count.min()}개, 중앙값 {crop_count.median():.0f}개, 최대 {crop_count.max()}개")

CELL = 100   # 잘라 낸 영역 하나를 넣을 칸의 크기(픽셀)
NCOLS = 12   # 한 줄에 넣을 칸 수
CELL_FONT = ImageFont.load_default(size=14)  # 칸 번호 글꼴

# 모든 bbox 영역을 잘라 칸 크기에 맞게 줄여 둔다. 이미지는 한 장씩 한 번만 연다.
crops = {}  # crop_df의 인덱스 → 줄인 영역 이미지
for file_name, rows in crop_df.groupby("file_name"):
    with Image.open(path_by_name[file_name]) as im:
        im = im.convert("RGB")
        for r in rows.itertuples():
            crop = im.crop((r.bbox_x, r.bbox_y, r.bbox_x + r.bbox_w, r.bbox_y + r.bbox_h))  # (왼쪽, 위, 오른쪽, 아래)
            crop.thumbnail((CELL - 4, CELL - 4))  # 비율을 유지한 채 칸 안에 들어가게 줄임
            crops[r.Index] = crop


def class_cells(cat_id):
    """한 클래스의 bbox 인덱스를 칸 순서대로 돌려준다. 3-4의 그림과 3-5의 표가 같은 순서를 쓴다."""
    return list(crop_df.index[crop_df["category_id"] == cat_id])


def class_montage(cat_id):
    """한 클래스의 잘라 낸 영역을 한 줄에 NCOLS개씩 붙인 이미지를 만든다. 칸마다 순서 번호를 적는다."""
    idx = class_cells(cat_id)
    nrows = (len(idx) + NCOLS - 1) // NCOLS
    canvas = Image.new("RGB", (NCOLS * CELL, nrows * CELL), "white")
    draw = ImageDraw.Draw(canvas)
    for k, i in enumerate(idx):
        c = crops[i]
        col, row = k % NCOLS, k // NCOLS
        canvas.paste(c, (col * CELL + (CELL - c.width) // 2, row * CELL + (CELL - c.height) // 2))  # 칸 가운데에 붙임
        draw.text((col * CELL + 3, row * CELL + 1), str(k), fill="red", font=CELL_FONT)
    return canvas

In [ ]:
CLASS_PAGE = 8  # 그림 하나에 넣을 클래스 수
class_ids = sorted(crop_count.index)

for start in range(0, len(class_ids), CLASS_PAGE):
    page_ids = class_ids[start:start + CLASS_PAGE]
    montages = [class_montage(cid) for cid in page_ids]
    heights = [m.height for m in montages]
    # 클래스마다 줄 수가 달라서, 묶음마다 높이를 줄 수에 맞춰 나눈다.
    fig, axes = plt.subplots(len(page_ids), 1, figsize=(12, sum(heights) / 100 + 0.45 * len(page_ids)),
                             gridspec_kw={"height_ratios": heights}, layout="constrained", squeeze=False)
    for ax, cid, m in zip(axes[:, 0], page_ids, montages):
        ax.imshow(m)
        ax.set_title(f"{cid} {class_names[cid]} (n={crop_count[cid]})", loc="left", fontsize=9)
        ax.axis("off")
    fig.suptitle(f"클래스별 bbox 영역 ({start + 1}~{start + len(page_ids)}번째 클래스 / {len(class_ids)}개)", fontsize=11)
    plt.show()

#### 3-4 결과

- 이미지 밖 bbox 1개를 뺀 762개, 56개 클래스의 영역을 잘라 모았다. 클래스별 bbox 수는 최소 3개, 중앙값 9개, 최대 153개다.

그림에서 보이는 것

같은 클래스의 다른 칸과 다르게 보이는 칸을 두 가지로 나눴다. 칸 번호는 그림의 빨간 숫자다.

1) 다른 알약이 잘렸거나, 알약이 없거나, 알약이 잘린 칸

| 클래스 | 칸 | 같은 클래스의 다른 칸 | 이 칸 |
|---|---|---|---|
| 3351 | 97 | HT 각인 주황색 원형 알약 | 알약 없이 바닥만 보임 |
| 3351 | 129 | HT 각인 주황색 원형 알약 | GIL 각인 흰색 원형 알약(31863 칸과 같은 모양) |
| 16548 | 3 | GB 각인 노란색 원형 알약 | S300 각인 갈색 타원형 알약(33009 칸과 같은 모양) |
| 20238 | 7 | 75 각인 분홍색 원형 알약 | HT 각인 주황색 원형 알약(3351 칸과 같은 모양) |
| 27733 | 10 | A1 각인 흰색 타원형 알약 | 337 각인 흰색 긴 알약(35206 칸과 같은 모양) |
| 35206 | 29 | 337 각인 흰색 긴 알약 | A1 각인 흰색 타원형 알약(27733 칸과 같은 모양) |
| 19232 | 8 | HM-CN 글자의 노란색 타원형 캡슐 전체 | 같은 캡슐인데 아래쪽이 칸 밖으로 잘림 |

2) 색과 모양은 같은데 각인이 다른 칸

| 클래스 | 칸 | 다른 칸의 각인 | 이 칸의 각인 |
|---|---|---|---|
| 3351 | 124 | HT | 가운데 분할선과 다른 글자 |
| 3483 | 9, 10, 11, 15, 16, 17 | G40 | SK |
| 22074 | 7 | LPL | 20으로 보이는 글자 |
| 30308 | 3, 4, 5 | 로고 | D2/850 |

- 1)은 bbox가 잘못된 알약이나 빈 곳에 놓였거나 bbox 크기가 알약과 맞지 않는 후보다. 3-3의 좌표가 같은 3쌍(3351 129, 16548 3, 20238 7)이 여기서도 보인다.
- 2)는 같은 알약의 앞면과 뒷면일 가능성이 있어서 따로 뒀다. 2-3에서 뒷면 사진이 4장 있었다.
- 그 밖에 3351의 109, 131, 140번 칸은 초점이 흐리지만 같은 HT 알약으로 보여서 넣지 않았다.

어떤 사진의 어떤 bbox인지, 앞뒷면인지는 3-5에서 확인한다.

### 3-5. 3-4에서 다르게 보인 칸의 원래 사진

3-4 결과의 두 표에 적은 칸이 어느 사진의 어느 bbox인지 찾는다.

- 1) 다른 알약이 잘렸거나, 알약이 없거나, 알약이 잘린 칸: 칸마다 사진 파일명, bbox, JSON 경로를 표로 보여 준다. 사진 전체에 bbox를 모두 그리고, 다르게 보인 칸의 bbox는 파란색으로 `클래스 (#칸 번호)`를 적어 구분한다.
- 2) 각인이 다른 칸: 그 칸 사진의 `drug_dir`(앞뒷면)과, 그 클래스 JSON의 `print_front`, `print_back`(앞면, 뒷면 식별 문자)을 표로 보여 준다. 각인이 다른 칸이 뒷면 사진(`drug_dir`이 뒷면)에 있고 그 각인이 `print_back`과 같다면, 같은 알약의 뒷면일 것이라고 보고 확인한다.

In [ ]:
# 3-4 그림에서 다르게 보인 칸: (클래스, 칸 번호, 보인 것)
SUSPECT_CELLS = [
    (3351, 97, "바닥만 보임"),
    (3351, 129, "GIL 흰색 알약"),
    (16548, 3, "S300 갈색 알약"),
    (20238, 7, "HT 주황색 알약"),
    (27733, 10, "337 흰색 알약"),
    (35206, 29, "A1 흰색 알약"),
    (19232, 8, "아래쪽이 잘림"),
]

suspect_rows = []
for cid, k, seen in SUSPECT_CELLS:
    idx = class_cells(cid)[k]  # 3-4 그림과 같은 순서
    r = ann_df.loc[idx]
    suspect_rows.append({"idx": idx, "category_id": cid, "칸": k, "보인 것": seen, "file_name": r["file_name"],
                         "각도": r["camera_la"], "bbox": r["bbox"], "json_path": r["json_path"]})
suspect_df = pd.DataFrame(suspect_rows).set_index("idx")
display(suspect_df)


def boxes_with_check(file_name):
    """이미지 하나의 bbox 목록. 다르게 보인 칸의 bbox는 파란색으로 '클래스 (#칸)'을 적는다."""
    boxes = []
    for r in crop_df[crop_df["file_name"] == file_name].itertuples():
        if r.Index in suspect_df.index:
            boxes.append((r.bbox_x, r.bbox_y, r.bbox_w, r.bbox_h, f"{r.category_id} (#{suspect_df.loc[r.Index, '칸']})", "#2a78d6"))
        else:
            boxes.append((r.bbox_x, r.bbox_y, r.bbox_w, r.bbox_h, str(r.category_id), "#e5484d"))
    return boxes


suspect_names = list(dict.fromkeys(suspect_df["file_name"]))  # 한 사진에 칸이 2개 이상일 수 있어서 순서를 지키며 중복 제거
show_thumbs(
    paths=[path_by_name[n] for n in suspect_names],
    captions=[f"{n.split('_')[0]}\n각도 {n.split('_')[5]}°, 앞뒷면 자리 {n.split('_')[3]}" for n in suspect_names],
    boxes=[boxes_with_check(n) for n in suspect_names],
    title="3-4에서 다르게 보인 칸의 원래 사진 (파란색: 다르게 보인 칸, 빨간색: 다른 bbox)",
    ncols=3,
)

In [ ]:
# 3-4 그림에서 각인이 다르게 보인 칸: (클래스, 칸 번호 목록)
OTHER_IMPRINT_CELLS = [(3351, [124]), (3483, [9, 10, 11, 15, 16, 17]), (22074, [7]), (30308, [3, 4, 5])]

json_by_path = {path.relative_to(TRAIN_ANN_DIR).as_posix(): data for path, data in loaded}  # 1-5에서 읽은 JSON

imprint_rows = []
for cid, cells in OTHER_IMPRINT_CELLS:
    all_idx = class_cells(cid)
    image0 = json_by_path[ann_df.loc[all_idx[0], "json_path"]]["images"][0]  # 같은 클래스는 알약 정보가 같다고 보고 첫 JSON에서 읽음
    back_cells = [k for k, i in enumerate(all_idx) if ann_df.loc[i, "drug_dir"] == "뒷면"]
    imprint_rows.append({
        "category_id": cid,
        "print_front": image0.get("print_front"),
        "print_back": image0.get("print_back"),
        "각인이 다르게 보인 칸": cells,
        "뒷면 사진의 칸": back_cells,
        "두 목록이 같음": sorted(cells) == back_cells,
    })
display(pd.DataFrame(imprint_rows))

#### 3-5 결과

1) 다른 알약이 잘렸거나, 알약이 없거나, 알약이 잘린 칸

7칸은 사진 6장에 있다(27733 #10과 35206 #29가 같은 사진). 아래는 그림에서 보이는 것이다.

| 칸 | 사진 | 보이는 것 |
|---|---|---|
| 3351 #97 | `K-003351-020014-020238` 90° | 3351 bbox `[185, 249, 188, 183]`가 HT 알약 바로 왼쪽의 빈 바닥에 있고, HT 알약에는 bbox가 없다. bbox 크기는 HT 알약과 비슷해서, 위치(x)만 어긋난 것으로 보인다. |
| 3351 #129 | `K-003351-029667-031863` 70° | 3-3의 좌표가 같은 쌍. GIL 알약 위에 31863 bbox와 같은 좌표로 있고, HT 알약에는 bbox가 없다. |
| 16548 #3 | `K-001900-016548-019607-033009` 70° | 3-3의 좌표가 같은 쌍. S300 알약 위에 33009 bbox와 같은 좌표로 있고, GB 노란색 알약에는 bbox가 없다. |
| 20238 #7 | `K-003351-020238-031863` 70° | 3-3의 좌표가 같은 쌍. HT 알약 위에 3351 bbox와 같은 좌표로 있고, 75 알약에는 bbox가 없다. |
| 27733 #10, 35206 #29 | `K-003483-025367-027733-035206` 75° | 27733 bbox가 337 알약에, 35206 bbox가 A1 알약에 있다. 두 bbox 모두 알약을 잘 감싸지만, 두 클래스가 서로 바뀌어 있다. |
| 19232 #8 | `K-003351-019232-029667` 70°(파일명의 앞뒷면 자리 1) | bbox `[523, 201, 232, 241]`가 캡슐의 위쪽만 감싸고, 아래쪽은 bbox 밖으로 나간다. bbox 높이가 작은 것으로 보인다. 3-6의 area 비교에서 다시 본다. |

2) 각인이 다른 칸

- JSON의 식별 문자는 아래와 같다.

| 클래스 | `print_front` | `print_back` | 많은 칸의 각인(3-4) | 다르게 보인 칸의 각인(3-4) |
|---|---|---|---|---|
| 3351 | I분할선Y | HT | HT | 분할선과 글자 |
| 3483 | SK | G40 | G40 | SK |
| 22074 | LPL | 20 | LPL | 20으로 보이는 글자 |
| 30308 | 마크 | D2/850 | 로고 | D2/850 |

- 네 클래스 모두, 다르게 보인 칸의 각인이 그 클래스의 다른 면 식별 문자와 같아 보인다. 그래서 다른 약이 아니라 같은 알약의 다른 면이 보인 것으로 보고, 라벨 오류가 아닌 것으로 본다.
- 처음에는 이 칸들이 뒷면 사진(`drug_dir`이 뒷면)에 있을 것이라고 보고 비교했는데, 같지 않았다. 3483, 22074, 30308은 뒷면 사진에 없고, 3351이 뒷면 사진에 있는 4칸(87, 88, 104, 105)은 3-4 그림에서 많은 칸과 같은 HT로 보였다.
- `drug_dir`은 사진 전체에 붙은 값이라, 사진 속 알약 하나하나가 어느 면을 위로 놓였는지는 알려 주지 않는다. 어느 면이 더 많이 보이는지도 클래스마다 다르다(3351, 3483은 `print_back`, 22074, 30308은 `print_front` 쪽이 많다).
- 한 클래스 안에 각인이 두 가지인 알약이 섞여 있다는 뜻이다.

### 3-6. area 불일치

- 대회 설명에서 `area`는 bbox의 면적이라서, bbox의 너비 × 높이와 같아야 한다. 다른 것을 찾는다.
- 3-5에서 다르게 보인 7칸도 `area`와 너비 × 높이가 같은지 따로 보여 준다. 3-5에서 19232 #8은 bbox 높이가 작아 보였다.
- 다르면 bbox 숫자가 틀렸는지 `area` 숫자가 틀렸는지 알 수 없다. 사람이 값을 옮겨 적다가 숫자 한 자리를 잘못 적었다고 가정하고, 숫자 한 자리만 바꿔서 맞는 쪽을 찾는다.
    - bbox 후보: 너비나 높이의 숫자 한 자리를 다른 숫자로 바꾸면 너비 × 높이가 `area`와 같아지는 경우
    - area 후보: `area`의 숫자 한 자리를 다른 숫자로 바꾸면 너비 × 높이와 같아지는 경우
- bbox 후보는 지금 bbox와 후보 bbox를 확대한 사진에 나란히 그려서, 어느 쪽이 알약을 감싸는지 본다. area 후보는 bbox가 바뀌지 않아서 그리지 않는다.
- 이 가정으로 후보가 나오지 않는 불일치는 따로 보여 준다.

In [ ]:
area_df = ann_df[["file_name", "category_id", "bbox_x", "bbox_y", "bbox_w", "bbox_h", "area", "json_path"]].copy()
area_df["w×h"] = area_df["bbox_w"] * area_df["bbox_h"]
area_bad = area_df[area_df["area"] != area_df["w×h"]].copy()
area_bad["area - w×h"] = area_bad["area"] - area_bad["w×h"]

print(f"area가 너비 × 높이와 다른 bbox: {len(area_bad)}개 / {len(area_df)}개")
if len(area_bad):
    display(area_bad[["file_name", "category_id", "bbox_w", "bbox_h", "w×h", "area", "area - w×h"]])

print()
print("3-5에서 다르게 보인 칸의 area")
suspect_area = area_df.loc[suspect_df.index, ["file_name", "category_id", "bbox_w", "bbox_h", "w×h", "area"]]
suspect_area.insert(2, "칸", suspect_df["칸"])
suspect_area["같음"] = suspect_area["area"] == suspect_area["w×h"]
display(suspect_area)


def one_digit_changes(value):
    """정수의 숫자 한 자리를 다른 숫자로 바꾼 값들을 돌려준다. 맨 앞자리는 0으로 바꾸지 않는다."""
    s = str(int(value))
    return {int(s[:i] + d + s[i + 1:]) for i in range(len(s)) for d in "0123456789" if d != s[i] and not (i == 0 and d == "0")}


cand_rows = []
for r in area_bad.itertuples():
    for field, value, other in [("너비", r.bbox_w, r.bbox_h), ("높이", r.bbox_h, r.bbox_w)]:
        for new in sorted(one_digit_changes(value)):
            if new * other == r.area:
                cand_rows.append({"idx": r.Index, "file_name": r.file_name, "category_id": r.category_id, "고칠 값": field, "지금": value, "후보": new})
    for new in sorted(one_digit_changes(r.area)):
        if new == r.bbox_w * r.bbox_h:
            cand_rows.append({"idx": r.Index, "file_name": r.file_name, "category_id": r.category_id, "고칠 값": "area", "지금": r.area, "후보": new})
area_cand = pd.DataFrame(cand_rows, columns=["idx", "file_name", "category_id", "고칠 값", "지금", "후보"])

print()
print(f"숫자 한 자리를 바꾸면 맞는 후보: {len(area_cand)}개 (bbox {int((area_cand['고칠 값'] != 'area').sum())}개, area {int((area_cand['고칠 값'] == 'area').sum())}개)")
if len(area_cand):
    display(area_cand)
no_cand = area_bad[~area_bad.index.isin(area_cand["idx"])]
print(f"후보가 없는 불일치: {len(no_cand)}개")
if len(no_cand):
    display(no_cand)

In [ ]:
MARGIN = 80  # 확대할 때 bbox 바깥으로 더 보여 줄 여백(픽셀)

bbox_cand = area_cand[area_cand["고칠 값"] != "area"]
bbox_cand = bbox_cand[bbox_cand["idx"].isin(crop_df.index)]  # 3-2의 이미지 밖 bbox는 사진 위에 그릴 수 없어서 뺀다
print(f"그릴 bbox 후보: {len(bbox_cand)}개")

for c in bbox_cand.to_dict("records"):  # 열 이름에 띄어쓰기가 있어서 itertuples 대신 dict로 꺼낸다
    x, y, w, h = ann_df.loc[c["idx"], "bbox"]
    new_w, new_h = (c["후보"], h) if c["고칠 값"] == "너비" else (w, c["후보"])
    with Image.open(path_by_name[c["file_name"]]) as im:
        img = im.convert("RGB")

    fig, axes = plt.subplots(1, 2, figsize=(9, 5), layout="constrained")
    panels = [(w, h, f"지금 bbox [{x}, {y}, {w}, {h}]", "#e5484d"),
              (new_w, new_h, f"후보 bbox [{x}, {y}, {new_w}, {new_h}]", "#2a78d6")]
    for ax, (bw, bh, title, color) in zip(axes, panels):
        ax.imshow(img)
        ax.add_patch(Rectangle((x, y), bw, bh, fill=False, edgecolor=color, linewidth=2))
        ax.set_xlim(max(0, x - MARGIN), min(img.width, x + max(w, new_w) + MARGIN))
        ax.set_ylim(min(img.height, y + max(h, new_h) + MARGIN), max(0, y - MARGIN))  # 이미지 좌표는 아래로 갈수록 y가 커져서 큰 값을 먼저 넣는다
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    fig.suptitle(f"{c['file_name']}\n{c['category_id']}: {c['고칠 값']} {c['지금']} → {c['후보']}, area {ann_df.loc[c['idx'], 'area']}", fontsize=10)
    plt.show()

#### 3-6 결과

- `area`가 너비 × 높이와 다른 bbox는 763개 중 6개다.
- 6개 모두 숫자 한 자리를 바꾸면 맞는 후보가 하나씩 나왔다. 후보가 없는 불일치는 없다.

| 클래스 | 사진 | 고칠 값 | 지금 → 후보 | 후보로 계산 |
|---|---|---|---|---|
| 12778 | `K-002483-012081-012778-025438` 90° | area | 161096 → 161036 | 317 × 508 = 161036 |
| 19232 | `K-003351-019232-029667` 70° (3-5의 #8) | 높이 | 241 → 341 | 232 × 341 = 79112 |
| 19232 | `K-003351-019232-035206` 90° | 너비 | 322 → 332 | 332 × 266 = 88312 |
| 3351 | `K-003351-021325-033880` 70° | 높이 | 182 → 192 | 189 × 192 = 36288 |
| 33880 | `K-003351-033880-038162` 70° | area | 129480 → 129780 | 309 × 420 = 129780 |
| 38162 | `K-003351-033880-038162` 70° | area | 62009 → 65009 | 251 × 259 = 65009 |

- 3-5의 7칸 중 `area`가 다른 것은 19232 #8 하나다. 나머지 6칸은 `area`가 너비 × 높이와 같다. `area`는 bbox의 크기만 검사해서, bbox가 다른 알약 위에 있거나 클래스가 바뀐 오류(3-5)는 이 방법으로 찾을 수 없다.
- 학습에서 bbox만 쓰면 `area` 값은 쓰이지 않는다. 그래도 `area`가 맞지 않는 것은 bbox 숫자가 틀렸다는 신호일 수 있다(19232 #8).

그림에서 보이는 것 (bbox 후보 3개)

| 사진 | 지금 bbox | 후보 bbox |
|---|---|---|
| 19232, `K-003351-019232-029667` 70° | 높이 241. 캡슐의 위쪽만 감싸고 아래쪽이 밖으로 나간다(3-5와 같은 bbox). | 높이 341. 캡슐 전체를 감싼다. |
| 19232, `K-003351-019232-035206` 90° | 너비 322. 캡슐 전체를 감싼다. | 너비 332. 캡슐 전체를 감싸고 오른쪽 여백이 조금 더 있다. 차이가 10픽셀이라 그림만으로는 어느 쪽이 맞는지 가리기 어렵다. |
| 3351, `K-003351-021325-033880` 70° | 높이 182. 알약을 감싸지만 아래 변이 알약 아래 끝에 거의 붙어 있다. | 높이 192. 알약을 감싸고 위아래 여백이 비슷하다. |

- `area` 후보 3개는 bbox가 그대로라서 그리지 않았다.

### 3-7. 결측치

- 결측치(missing value)는 값이 있어야 할 자리에 값이 없는 것이다. 1-4에서 JSON 하나에 값이 빈 글자인 항목이 여럿 있었다. 모든 JSON에서 항목마다 값이 비어 있는 경우를 센다.
- 비어 있는 경우는 세 가지로 나눠 센다. `None`(JSON의 `null`), 빈 글자(`""`, 공백만 있는 글자 포함), 빈 목록(`[]`)이다.
- 비어 있는 값이 하나라도 있는 항목만 보여 준다. JSON 763개 모두에서 비어 있으면 그 항목은 이 데이터에서 쓸 수 없는 항목이다.
- 변수 이름은 3-1의 `missing_df`(라벨 누락 후보)와 겹치지 않게 `blank_df`로 한다.

In [ ]:
blank_rows = []
for block in ["images", "annotations", "categories"]:
    keys = list(loaded[0][1][block][0].keys())  # 1-5에서 목록별 key 구성이 한 가지로 같았다
    for key in keys:
        values = [d[block][0].get(key) for _, d in loaded]
        n_none = sum(v is None for v in values)
        n_empty_str = sum(isinstance(v, str) and v.strip() == "" for v in values)
        n_empty_list = sum(isinstance(v, list) and len(v) == 0 for v in values)
        n_blank = n_none + n_empty_str + n_empty_list
        if n_blank:
            blank_rows.append({"목록": block, "항목": key, "None": n_none, "빈 글자": n_empty_str, "빈 목록": n_empty_list,
                               "비어 있는 JSON 수": n_blank, "모두 비어 있음": n_blank == len(loaded)})

blank_df = pd.DataFrame(blank_rows)
print(f"JSON {len(loaded)}개 기준, 비어 있는 값이 있는 항목: {len(blank_df)}개")
if len(blank_df):
    display(blank_df.sort_values(["모두 비어 있음", "비어 있는 JSON 수"], ascending=False).reset_index(drop=True))

#### 3-7 결과

- 비어 있는 값이 있는 항목은 23개다. `annotations`에서는 `segmentation` 하나뿐이고, `categories`에는 없다. 나머지 22개는 `images` 항목이다.
- 학습에 쓰는 `bbox`, `category_id`, `file_name`, `width`, `height`는 목록에 없다. 즉 비어 있는 JSON이 없다.
- 763개 모두 비어 있는 항목은 5개다. `mark_code_front`, `mark_code_back`, `mark_code_back_anal`, `mark_code_back_img`, `segmentation`이다. 이 데이터에서는 쓸 수 없다. `segmentation`은 bbox만 쓰는 이번 과제와는 관계없다.
- `None`이 있는 항목은 18개이고, 모두 `None`이 18개씩이다. 알약 정보 항목(`chart`, `drug_shape`, `color_class1`, `print_front` 등)이다.
- 빈 글자가 많은 항목은 `color_class2`(두 번째 색), `line_front`, `line_back`(분할선), `print_back`(뒷면 식별 문자), `mark_code_*`(마크), `di_company_mf`(위탁제조사)다. 항목 뜻으로 보면 그 알약에 해당 사항이 없어서 비어 있을 수도 있다(한 가지 색, 분할선 없음 등). 이 표만으로는 "없음"인지 "적지 않음"인지 구분할 수 없다.
- 제품명 영어(`dl_name_en`) 14개, 성분명(`dl_material`) 9개는 빈 글자다. 알약마다 값이 있어야 할 항목이다.

`None`이 18개씩 나온 것이 같은 JSON 18개인지, 어느 클래스인지는 이 표로 알 수 없어서 아래에서 본다.

#### 3-7 추가. `None`이 있는 JSON

- `images[0]`에 `None`이 하나라도 있는 JSON을 모은다.
- `None`인 항목 구성이 몇 가지인지 센다. 한 가지면 같은 JSON들에서 같은 항목들이 함께 비어 있는 것이다.
- 클래스별로, `None`이 있는 JSON 수와 그 클래스의 전체 JSON 수를 비교한다. 두 수가 같으면 그 클래스는 어느 JSON에서도 알약 정보를 얻을 수 없다.

In [ ]:
none_rows = []
for path, d in loaded:
    image0 = d["images"][0]
    none_keys = tuple(k for k, v in image0.items() if v is None)
    if none_keys:
        none_rows.append({"json_path": path.relative_to(TRAIN_ANN_DIR).as_posix(),
                          "category_id": d["annotations"][0]["category_id"],
                          "dl_name": image0.get("dl_name"),
                          "None 항목": none_keys})
none_df = pd.DataFrame(none_rows)

print(f"images[0]에 None이 있는 JSON: {len(none_df)}개 / {len(loaded)}개")
if len(none_df):
    key_sets = Counter(none_df["None 항목"])
    print(f"None 항목 구성: {len(key_sets)}가지")
    for keys, n in key_sets.items():
        print(f"  JSON {n}개, 항목 {len(keys)}개: {', '.join(keys)}")

    class_json_count = ann_df["category_id"].value_counts()  # 클래스별 전체 JSON 수
    none_by_class = none_df.groupby(["category_id", "dl_name"]).size().rename("None이 있는 JSON 수").reset_index()
    none_by_class["클래스 전체 JSON 수"] = none_by_class["category_id"].map(class_json_count)
    display(none_by_class)

#### 3-7 추가 결과

- `None`이 있는 JSON 18개는 모두 16548(가바토파정 100mg)이고, 이 클래스의 JSON 18개 전부다.
- `None`인 항목 구성은 한 가지다. 18개 JSON 모두 같은 18개 항목(`chart`, `drug_shape`, `print_front`, `print_back`, `color_class1`, `line_front` 등 알약 외형 정보와 `img_key`, 날짜 항목)이 `None`이다. 나머지 32개 항목에는 `None`이 없다.
- 그래서 16548은 이 데이터의 어느 JSON에서도 모양, 색, 식별 문자 같은 외형 정보를 얻을 수 없다. 사진에는 알약이 있어서, 3-4 그림에서 16548 칸은 GB 각인 노란색 원형 알약으로 보였다.

### 3-8. 오류 목록과 고치는 방안

3-1~3-7에서 찾은 오류를 한곳에 모으고, 고친다면 어떻게 고칠 수 있는지 적는다. 이 노트북에서는 고치지 않는다.

#### 오류 목록

- 근거의 "숫자"는 값을 비교해서 나온 것이고, "그림"은 사진을 보고 판단한 것이다.
- 사진은 조합 이름과 각도로 적는다.

| 종류 | 찾은 곳 | 사진 | 클래스 | 지금 상태 | 근거 |
|---|---|---|---|---|---|
| 라벨 누락 | 3-1 | `K-003351-013900-021325` 70° | 3351 | 사진의 알약에 bbox가 없음 | 숫자, 그림 |
| 라벨 누락 | 3-1 | `K-003351-013900-036637` 70° | 3351 | 〃 | 숫자, 그림 |
| 라벨 누락 | 3-1 | `K-003351-032310-038162` 70° | 3351 | 〃 | 숫자, 그림 |
| 라벨 누락 | 3-1 | `K-003351-035206-041768` 70° | 3351 | 〃 | 숫자, 그림 |
| 라벨 누락 | 3-1 | `K-003351-020014-022074` 90° | 20014 | 〃 | 숫자, 그림 |
| 라벨 누락 | 3-1 | `K-003351-021325-032310` 90° | 32310 | 〃 | 숫자, 그림 |
| 라벨 누락 | 3-1 | `K-003351-033880-038162` 75° | 33880 | 〃 | 숫자, 그림 |
| 라벨 누락 | 3-1 | `K-003544-004543-012247-016548` 90° | 4543 | 〃 | 숫자, 그림 |
| 이미지 밖 bbox | 3-2 | `K-003351-016262-018357` 75° | 18357 | x가 6567로 이미지 너비(976)를 넘음 | 숫자, 그림 |
| 다른 알약 위 bbox | 3-3, 3-5 | `K-001900-016548-019607-033009` 70° | 16548 | 33009와 같은 좌표로 S300 알약 위에 있음. GB 알약에 bbox 없음 | 숫자, 그림 |
| 다른 알약 위 bbox | 3-3, 3-5 | `K-003351-020238-031863` 70° | 20238 | 3351과 같은 좌표로 HT 알약 위에 있음. 75 알약에 bbox 없음 | 숫자, 그림 |
| 다른 알약 위 bbox | 3-3, 3-5 | `K-003351-029667-031863` 70° | 3351 | 31863과 같은 좌표로 GIL 알약 위에 있음. HT 알약에 bbox 없음 | 숫자, 그림 |
| 빈 바닥 bbox | 3-5 | `K-003351-020014-020238` 90° | 3351 | HT 알약 왼쪽의 빈 바닥에 있음. HT 알약에 bbox 없음 | 그림 |
| 클래스 바뀜 | 3-5 | `K-003483-025367-027733-035206` 75° | 27733 | 337 알약 위에 있음 | 그림 |
| 클래스 바뀜 | 3-5 | `K-003483-025367-027733-035206` 75° | 35206 | A1 알약 위에 있음 | 그림 |
| bbox 크기 | 3-5, 3-6 | `K-003351-019232-029667` 70° | 19232 | 높이 241. 캡슐 아래쪽이 bbox 밖으로 나감 | 숫자, 그림 |
| bbox 크기 | 3-6 | `K-003351-019232-035206` 90° | 19232 | 너비 322. area와 맞지 않음 | 숫자 |
| bbox 크기 | 3-6 | `K-003351-021325-033880` 70° | 3351 | 높이 182. area와 맞지 않음 | 숫자 |
| area 값 | 3-6 | `K-002483-012081-012778-025438` 90° | 12778 | area 161096 | 숫자 |
| area 값 | 3-6 | `K-003351-033880-038162` 70° | 33880 | area 129480 | 숫자 |
| area 값 | 3-6 | `K-003351-033880-038162` 70° | 38162 | area 62009 | 숫자 |

- 오류는 bbox 21개(새로 추가할 것 8개, 고칠 것 13개)이고, 사진 19장에 있다.

#### 고치는 방안

| 종류 | 수 | 고치는 방안 |
|---|---|---|
| 라벨 누락 | 8 | bbox가 없는 알약에 bbox를 새로 그려 JSON을 추가한다. 이미지 정보(파일명, 크기, 촬영 정보)는 같은 사진의 다른 JSON에서, 알약 정보(제품명 등)는 같은 클래스의 다른 JSON에서 가져온다. bbox 좌표는 사진을 보고 정한다. |
| 이미지 밖 bbox | 1 | x를 567로 고친다. 3-2에서 숫자 6이 하나 더 들어간 것으로 보인 후보다. |
| 다른 알약 위 bbox | 3 | 잘못 놓인 클래스의 bbox를 bbox가 없는 알약 위로 옮긴다. 좌표는 사진을 보고 새로 정한다. |
| 빈 바닥 bbox | 1 | bbox를 HT 알약 위로 옮긴다. bbox 크기는 알약과 비슷해 보여서 위치(x, y)만 고치는 후보다. |
| 클래스 바뀜 | 2 | 두 JSON의 bbox와 area를 서로 바꾼다. |
| bbox 크기 | 3 | area와 맞는 후보 값(341, 332, 192)으로 고친다. 19232 70°는 그림으로도 후보가 맞아 보였다. 나머지 2개는 area만 근거라서 사진을 다시 보고 정한다. |
| area 값 | 3 | area를 너비 × 높이(161036, 129780, 65009)로 고친다. bbox만 쓰는 학습에는 영향이 없다. |

- 원본 JSON은 그대로 두고, 고칠 목록(사진, 클래스, 고칠 값)을 표로 정리해서 사본에 적용한다. 무엇을 고쳤는지 기록이 남고, 잘못 고친 것을 되돌릴 수 있다.
- 좌표를 새로 정해야 하는 것(라벨 누락, 다른 알약 위 bbox, 빈 바닥 bbox)은 숫자로 계산할 수 없어서 사람이 사진을 보고 정해야 한다.
- 몇 픽셀 차이도 고칠 이유가 있다. 평가 지표 mAP@[0.75:0.95]는 IoU 기준을 0.75부터 0.95까지 올려 가며 맞았는지 센다. 한 bbox가 다른 bbox 안에 들어가 있으면 IoU는 작은 넓이 ÷ 큰 넓이라서, 높이 182와 192의 차이만으로 IoU는 182 ÷ 192 ≈ 0.948이 된다. 학습 라벨이 이만큼 어긋나 있으면 모델이 배우는 bbox도 어긋날 수 있다.

#### 결측치

| 내용 | 고치는 방안 |
|---|---|
| 학습에 쓰는 값(`bbox`, `category_id`, `file_name`, `width`, `height`)에는 결측치가 없다. | 검출 모델 학습을 위해 고칠 것은 없다. |
| 16548의 JSON 18개는 알약 외형 정보 18개 항목이 모두 `None`이다. | 외형 정보가 필요하면(앱에서 약 정보를 보여 줄 때 등) `item_seq`(품목기준코드, 비어 있는 JSON 없음)로 외부 의약품 정보에서 찾아 채울 수 있다. |
| 763개 모두 비어 있는 항목이 5개(`mark_code_front`, `mark_code_back`, `mark_code_back_anal`, `mark_code_back_img`, `segmentation`)다. | 쓰지 않는다. |

#### 오류가 아닌 것

| 내용 | 찾은 곳 | 판단 |
|---|---|---|
| 33880과 35206 bbox의 가장자리 겹침(`K-003351-033880-035206` 75°, 90°) | 3-3 | 두 알약이 가까이 놓여 있고, 각 bbox는 자기 알약을 감싸는 것으로 보인다. |
| 한 클래스 안에서 각인이 다르게 보인 칸(3351, 3483, 22074, 30308) | 3-4, 3-5 | 다르게 보인 각인이 그 클래스의 다른 면 식별 문자와 같아 보인다. 같은 알약의 다른 면으로 본다. |

### 3-9. 정리

#### 확인한 것

- 라벨 오류: bbox 21개(새로 추가할 것 8개, 고칠 것 13개)가 사진 19장에 있다. train 232장의 약 8%다. 종류는 라벨 누락, 이미지 밖 bbox, 다른 알약 위 bbox, 빈 바닥 bbox, 클래스 바뀜, bbox 크기, area 값이다.
- 찾는 방법
    - 라벨 누락, 이미지 밖 bbox, area 불일치는 숫자 비교로 찾았다.
    - 다른 알약 위 bbox는 좌표가 같은 쌍으로 찾았지만, 어느 클래스가 잘못 놓였는지는 사진을 봐야 알 수 있었다.
    - 빈 바닥 bbox와 클래스 바뀜은 숫자로 드러나지 않았고, 클래스별로 bbox 영역을 모아 보고 찾았다.
- 그래서 숫자 검사로 드러나지 않는 오류가 더 있을 수 있다. 3-4는 bbox 영역을 작게 줄여 본 것이라, 위치가 조금만 어긋난 bbox는 찾기 어렵다.
- 오류가 아닌 것으로 본 것: 가까이 놓인 알약의 bbox 가장자리 겹침, 한 클래스 안에 각인이 두 가지인 것(같은 알약의 앞면과 뒷면으로 보임).
- 결측치: 학습에 쓰는 값에는 없다. 16548은 JSON 18개 모두 알약 외형 정보가 `None`이다. 763개 모두 비어 있는 항목이 5개다.

#### 뒤 섹션에서 볼 것

- 3-4에서 클래스별 bbox 수가 최소 3개, 최대 153개로 차이가 컸다. 클래스 분포는 4. 클래스에서 본다.
- 한 클래스 안에 각인이 두 가지인 알약이 있었다. 클래스별 외형 정보는 4. 클래스에서 본다.
- 1차 학습에 적용한 라벨 수정은 8-1에 정리한다.

## 4. 클래스

클래스(알약 종류)마다 데이터가 얼마나 있는지, 어떤 조합에 나오는지, 어떻게 생겼는지 본다.

| 단계 | 하는 일 | 이유 |
|---|---|---|
| 4-1 | 클래스 목록 | category_id와 약 이름이 일대일인지 보고, 클래스마다 bbox 수, 사진 수, 조합 수를 한 표로 만든다. 뒤 단계는 이 표를 쓴다. |
| 4-2 | 클래스 분포 | 클래스 불균형이 얼마나 심한지 봐야 학습할 때의 대처(가중치, 증강, 보충 데이터)를 정할 수 있다. |
| 4-3 | 클래스별 조합 수 | train과 val을 조합 단위로 나눌 때, 조합이 1개뿐인 클래스는 한쪽에만 들어간다. 그런 클래스가 몇 개인지 본다. |
| 4-4 | 외형 정보 | 모양과 색이 같은 클래스는 모델이 헷갈리기 쉬운 후보다. |

- 개수는 원본 라벨(`ann_df`) 기준이다. 3-8의 오류를 고치면 일부 클래스의 수가 조금 바뀔 수 있다. 예를 들어 라벨 누락 8개를 추가하면 3351의 bbox가 4개 늘어난다.

### 4-1. 클래스 목록

- 1-7에서는 JSON 하나 안에서 클래스 번호와 약 이름이 서로 맞는지 봤다. 여기서는 JSON 전체에서 category_id 하나에 약 이름이 하나뿐인지, 약 이름 하나에 category_id가 하나뿐인지 본다. 한 번호에 이름이 여럿이면 다른 약이 같은 번호로 묶인 것이다.
- 한 사진에 같은 클래스의 bbox가 2개 이상인 경우를 센다. 없으면 클래스마다 bbox 수와 사진 수가 같다.
- 클래스마다 bbox 수, 사진 수, 조합 수를 한 표(`class_df`)로 만든다. 조합 수는 그 클래스가 나오는 조합 이름의 종류 수다. 1-8에서 본 파일명 규칙대로, 같은 조합 이름의 사진은 같은 알약들을 각도와 앞뒷면만 바꿔 찍은 것이다.

In [ ]:
names_per_id = ann_df.groupby("category_id")["dl_name"].nunique()   # category_id 하나에 붙은 약 이름 수
ids_per_name = ann_df.groupby("dl_name")["category_id"].nunique()   # 약 이름 하나에 붙은 category_id 수
same_class_in_image = ann_df.groupby(["file_name", "category_id"]).size()

print(f"클래스(category_id): {ann_df['category_id'].nunique()}개, 약 이름(dl_name): {ann_df['dl_name'].nunique()}개")
print(f"약 이름이 2개 이상인 category_id: {int((names_per_id > 1).sum())}개")
print(f"category_id가 2개 이상인 약 이름: {int((ids_per_name > 1).sum())}개")
print(f"한 사진에 같은 클래스 bbox가 2개 이상인 경우: {int((same_class_in_image > 1).sum())}건")

class_df = (
    ann_df.groupby("category_id")
    .agg(**{"dl_name": ("dl_name", "first"), "bbox 수": ("file_name", "size"),
            "사진 수": ("file_name", "nunique"), "조합 수": ("combo", "nunique")})
    .sort_values(["bbox 수", "조합 수"], ascending=False)
)

print()
print("클래스 목록 (bbox가 많은 순)")
with pd.option_context("display.max_rows", None):  # 56행을 줄이지 않고 모두 보여 준다
    display(class_df)

#### 4-1 결과

- 클래스(category_id)는 56개, 약 이름도 56개다. category_id와 약 이름은 일대일이다.
- 한 사진에 같은 클래스의 bbox가 2개 이상인 경우는 없다. 그래서 표에서 클래스마다 bbox 수와 사진 수가 같다. 한 사진에 같은 약이 두 알 놓인 경우가 없다는 뜻이다.
- 3351(일양하이트린정 2mg)은 사진 153장, 조합 89개에 나온다. train 사진 232장의 66%, 조합 114개의 78%다.
- 표 아래쪽 17개 클래스는 bbox 3개, 조합 1개다.

### 4-2. 클래스 분포

- 클래스 불균형(class imbalance)은 클래스마다 학습 데이터 수가 크게 다른 것이다. 심하면 모델이 데이터가 많은 클래스 위주로 배우고, 적은 클래스는 잘 맞히지 못할 수 있다.
- 클래스별 bbox 수의 최소, 중앙값, 최대와 최대 ÷ 최소를 계산한다. 가장 많은 클래스가 전체 bbox에서 차지하는 비율도 본다.
- bbox 수 구간(1~5, 6~10, 11~20, 21~50, 51 이상)별로 클래스 수를 센다. bbox가 적은 클래스가 몇 개인지 보기 위해서다.
- 클래스별 bbox 수를 많은 순으로 막대그래프로 그리고, 중앙값을 점선으로 표시한다.

In [ ]:
counts = class_df["bbox 수"]
top_id = counts.idxmax()
print(f"클래스별 bbox 수: 최소 {counts.min()}, 중앙값 {counts.median():g}, 최대 {counts.max()}, 최대 ÷ 최소 {counts.max() / counts.min():.1f}")
print(f"가장 많은 클래스: {top_id} ({class_df.loc[top_id, 'dl_name']}), 전체 bbox {counts.sum()}개 중 {counts.max()}개 ({counts.max() / counts.sum():.1%})")

count_bins = pd.cut(counts, bins=[0, 5, 10, 20, 50, np.inf], labels=["1~5", "6~10", "11~20", "21~50", "51 이상"])
display(count_bins.value_counts(sort=False).rename_axis("bbox 수").reset_index(name="클래스 수"))

fig, ax = plt.subplots(figsize=(15, 4.5), layout="constrained")
x = np.arange(len(counts))
ax.bar(x, counts.to_numpy(), width=0.7, color="#2a78d6")
ax.axhline(counts.median(), color="#555555", linewidth=1, linestyle="--")
ax.text(len(counts) - 0.5, counts.median(), f"중앙값 {counts.median():g}", ha="right", va="bottom", fontsize=9, color="#555555")
ax.set_xticks(x, counts.index.astype(str), rotation=90, fontsize=8)
ax.set_xlim(-0.6, len(counts) - 0.4)
ax.set_xlabel("category_id (bbox가 많은 순)")
ax.set_ylabel("bbox 수")
ax.set_title("클래스별 bbox 수")
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)  # 격자선을 막대 뒤로 보낸다
plt.show()

#### 4-2 결과

- 클래스별 bbox 수는 최소 3개, 중앙값 9개, 최대 153개다. 가장 많은 클래스가 가장 적은 클래스의 51배다.
- 3351 하나가 전체 bbox 763개 중 153개(20.1%)다. 다음으로 많은 3483(45개), 35206(37개)과도 차이가 크고, 나머지 53개 클래스는 23개 이하다.
- bbox가 5개 이하인 클래스가 18개(56개의 32%)다. 그중 17개는 3개이고, 1개(4543)는 5개다.
- 클래스 수는 11~20개 구간이 21개로 가장 많고, 21~50개 구간은 5개, 51개 이상은 3351 하나다.
- 이대로 학습하면 3351 위주로 배우고, bbox가 3개뿐인 클래스는 잘 맞히지 못할 수 있다. 대처 방법(가중치, 증강, 보충 데이터)은 이 EDA에서 정하지 않는다.

### 4-3. 클래스별 조합 수

- 같은 알약들을 찍은 사진이 train과 val에 함께 들어가면, val 점수가 처음 보는 사진에서의 성능보다 높게 나올 수 있다. 이것을 데이터 누수(data leakage)라고 한다.
- 그래서 train과 val을 조합 단위로 나누는 방법을 생각할 수 있다. 이때 조합이 1개뿐인 클래스는 train과 val 중 한쪽에만 들어간다.
- 조합 수별로 클래스 수를 세고, 조합이 1개뿐인 클래스는 약 이름, 사진 수, 조합 이름을 표로 보여 준다.

In [ ]:
combo_counts = class_df["조합 수"]
display(combo_counts.value_counts().sort_index().rename_axis("조합 수").reset_index(name="클래스 수"))

single = class_df[combo_counts == 1]
print(f"조합이 1개뿐인 클래스: {len(single)}개 / {len(class_df)}개")
print(f"조합이 2개 이상인 클래스: {int((combo_counts >= 2).sum())}개")
if len(single):
    single_combo = ann_df[ann_df["category_id"].isin(single.index)].groupby("category_id")["combo"].first()
    display(single.assign(조합=single_combo)[["dl_name", "사진 수", "조합"]])

#### 4-3 결과

- 조합이 1개뿐인 클래스는 17개(56개의 30%)이고, 모두 사진이 3장이다. 4-2에서 bbox가 3개인 17개 클래스와 같은 클래스다.
- 이 17개 클래스는 조합 13개에 있다. 4개 조합에는 이런 클래스가 2개씩 있다(`K-002483-003743-012778-013395`, `K-002483-019552-022362-025438`, `K-001900-016548-021771-027926`, `K-001900-016551-029345-033208`). 13개 조합 모두 조합 이름에 알약 코드가 4개다.
- 조합이 2개인 클래스는 7개다. 조합 단위로 train과 val에 하나씩 나누면 양쪽에 조합이 1개씩만 남는다.
- 조합이 9~11개인 클래스도 17개다. 3351은 조합 89개에 나와서 다른 클래스(최대 16개)보다 훨씬 많다.
- 조합 단위로 나누면 조합이 1개뿐인 17개 클래스는 train과 val 중 한쪽에만 들어간다.
    - val에 넣으면 그 클래스는 학습하지 못한다.
    - train에 넣으면 val에서 그 클래스의 성능을 볼 수 없다.
- 같은 조합의 사진끼리 얼마나 비슷한지는 6. 조합과 촬영 각도에서 본다. 1차 학습에 쓴 train/val 분할은 8-2에 정리한다.

### 4-4. 외형 정보

- JSON의 `images`에는 알약의 외형 정보가 있다. 모양(`drug_shape`), 색(`color_class1`, `color_class2`), 식별 문자(`print_front`, `print_back`), 정제 분류(`form_code_name`), 크기(`leng_long` 장축, `leng_short` 단축, 단위 mm)를 본다.
- 먼저 같은 클래스의 JSON끼리 이 값들이 같은지 센다. 같으면 클래스마다 첫 JSON의 값을 쓴다. 16548은 3-7에서 외형 정보가 모두 `None`이었다.
- 클래스별 외형 정보를 표로 보여 주고, 모양과 색의 종류별 클래스 수를 센다.
- 모양과 색(`color_class1`)이 같은 클래스를 묶는다. 같은 묶음의 클래스는 식별 문자 말고는 구분할 단서가 적어서, 모델이 헷갈리기 쉬운 후보다.
- 클래스마다 bbox 영역 한 장씩을 잘라 모양, 색 순서로 늘어놓는다. JSON의 외형 정보와 사진 속 알약이 맞는지, 같은 묶음의 클래스가 실제로 비슷하게 생겼는지 눈으로 본다.

In [ ]:
SHAPE_FIELDS = ["drug_shape", "color_class1", "color_class2", "print_front", "print_back", "form_code_name", "leng_long", "leng_short"]

image_info = pd.DataFrame([
    {"category_id": d["annotations"][0]["category_id"], **{f: d["images"][0].get(f) for f in SHAPE_FIELDS}}
    for _, d in loaded
])

# 같은 클래스의 JSON끼리 값이 몇 가지인지. None도 한 가지 값으로 센다.
n_values = image_info.groupby("category_id")[SHAPE_FIELDS].nunique(dropna=False)
print("같은 클래스 안에서 값이 2가지 이상인 클래스 수:")
print((n_values > 1).sum().to_dict())

shape_df = class_df[["dl_name"]].join(image_info.groupby("category_id")[SHAPE_FIELDS].first())
shape_df = shape_df.sort_values(["drug_shape", "color_class1"], na_position="last")
print()
print("클래스별 외형 정보 (모양, 색 순)")
with pd.option_context("display.max_rows", None):
    display(shape_df)

print("모양별 클래스 수")
display(shape_df["drug_shape"].value_counts(dropna=False).rename_axis("drug_shape").reset_index(name="클래스 수"))
print("색(color_class1)별 클래스 수")
display(shape_df["color_class1"].value_counts(dropna=False).rename_axis("color_class1").reset_index(name="클래스 수"))

# 모양과 색이 같은 클래스 묶음. 외형 정보가 없는 클래스는 뺀다.
look_rows = []
for (shape, color), g in shape_df.dropna(subset=["drug_shape", "color_class1"]).groupby(["drug_shape", "color_class1"]):
    if len(g) >= 2:
        look_rows.append({"drug_shape": shape, "color_class1": color, "클래스 수": len(g),
                          "클래스 (식별 문자 앞/뒤)": ", ".join(f"{cid} ({r.print_front}/{r.print_back})" for cid, r in g.iterrows())})
look_df = pd.DataFrame(look_rows)
print(f"모양과 색이 같은 클래스 묶음: {len(look_df)}개, 들어 있는 클래스: {int(look_df['클래스 수'].sum()) if len(look_df) else 0}개")
if len(look_df):
    display(look_df.sort_values("클래스 수", ascending=False).reset_index(drop=True))

In [ ]:
LOOK_CELL = 160  # 클래스마다 잘라 낸 영역을 줄일 크기(픽셀)
LOOK_COLS = 8

look_ids = list(shape_df.index)  # 모양, 색 순서
nrows = (len(look_ids) + LOOK_COLS - 1) // LOOK_COLS
fig, axes = plt.subplots(nrows, LOOK_COLS, figsize=(2 * LOOK_COLS, 2.5 * nrows), layout="constrained", squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, cid in zip(axes.flat, look_ids):
    r = crop_df.loc[class_cells(cid)[0]]  # 클래스의 첫 칸(3-4 그림의 0번 칸)
    with Image.open(path_by_name[r["file_name"]]) as im:
        crop = im.convert("RGB").crop((r["bbox_x"], r["bbox_y"], r["bbox_x"] + r["bbox_w"], r["bbox_y"] + r["bbox_h"]))
    crop.thumbnail((LOOK_CELL, LOOK_CELL))
    ax.imshow(crop)
    info = shape_df.loc[cid]
    look = "정보 없음" if pd.isna(info["drug_shape"]) else f"{info['drug_shape']} {info['color_class1']}"
    ax.set_title(f"{cid}\n{look}", fontsize=8)
fig.suptitle("클래스별 bbox 영역 한 장씩 (모양, 색 순)", fontsize=11)
plt.show()

#### 4-4 결과

- 같은 클래스의 JSON끼리는 8개 외형 항목의 값이 모두 같다. 그래서 클래스마다 첫 JSON의 값을 써도 된다.
- 16548은 모양, 색, 식별 문자, 정제 분류가 비어 있고(3-7의 `None`), 장축과 단축은 0이다. 0은 3-7에서 빈 값으로 세지 않았지만, 알약 크기가 0일 수는 없어서 값이 없는 것과 같다.
- 모양은 타원형 23개, 원형 17개, 장방형 13개가 대부분이고, 육각형과 팔각형이 1개씩이다.
- 색(`color_class1`)은 하양 19개, 노랑 10개, 분홍 10개, 갈색 6개, 주황 5개 순이다. 16688은 `노랑, 투명`처럼 두 색이 한 값에 들어 있다.
- 장축은 6.1mm(19607)부터 24.0mm(16688)까지다(16548의 0 제외).
- 외형 정보가 있는 55개 클래스 중 49개(89%)가 모양과 색이 같은 다른 클래스와 묶인다. 묶음은 12개이고, 원형 하양과 장방형 하양이 7개씩으로 가장 크다. 묶이지 않는 클래스는 6개(25438, 27777, 41768, 16688, 31885, 19861)다.

그림에서 보이는 것

- 각인이 읽히는 칸은 대부분 JSON의 `print_front`나 `print_back` 중 하나와 같아 보인다(예: 29667 Kowa 203, 3483 G40, 27993 ZLT-100, 24850 Noltec 10).
- 16548 칸은 GB 각인의 노란색 원형 알약으로 보인다. JSON에는 정보가 없지만, 사진으로는 원형 노랑 묶음(29667, 21771)과 비슷해 보인다.
- JSON의 색과 사진에서 보이는 색이 다른 클래스가 있다. 19232, 32310, 18357은 JSON에서 갈색인데, 사진에서는 노란색 연질캡슐로 보인다. 같은 갈색 묶음의 12778, 19552(붉은 갈색 정제), 33009(진한 갈색 정제)와는 달라 보인다.
- 사진에서 서로 비슷해 보이는 클래스
    - 19232, 32310, 18357: 노란색 타원형 연질캡슐. 모양과 색이 거의 같고 글자만 다르다. 32310 칸에서는 글자가 보이지 않는다.
    - 원형 하양 7개(16232, 22074, 31863, 3544, 19607, 4543, 3743): 4543은 파란 글씨가 인쇄돼 있고, 나머지는 새긴 각인만 다르다.
    - 장방형 하양 정제 6개(3832, 1900, 12081, 13395, 22362, 27993): 새긴 각인만 다르다.
- 칸마다 알약이 놓인 방향이 달라서 글자가 여러 방향으로 돌아가 있다.
- 칸마다 따로 줄여서 그렸기 때문에, 칸 사이의 크기는 비교할 수 없다.

### 4-5. 정리

#### 확인한 것

- 클래스: 56개이고, category_id와 약 이름은 일대일이다. 한 사진에 같은 약은 한 알씩만 있다.
- 불균형
    - 3351 하나가 bbox의 20.1%이고, train 사진의 66%에 나온다.
    - bbox가 3개뿐인 클래스가 17개다.
    - 가장 많은 클래스와 가장 적은 클래스는 51배 차이다.
- 조합: bbox가 3개뿐인 17개 클래스(30%)는 조합도 1개뿐이다. train과 val을 조합 단위로 나누면 이 클래스들은 한쪽에만 들어간다.
- 외형
    - JSON의 외형 정보는 클래스 안에서 일정하다.
    - 외형 정보가 있는 55개 클래스 중 49개가 모양과 색이 같은 다른 클래스와 묶인다. 이 클래스들은 식별 문자가 주요 구분 단서다.
    - 그림에서는 JSON의 색과 사진의 색이 다르게 보이는 클래스가 있었다.
    - 16548은 외형 정보가 없다(`None`, 크기 0).

#### 정할 것 (이 EDA에서는 정하지 않는다)

- 클래스 불균형에 대한 대처(가중치, 샘플링, 증강, 보충 데이터)

#### 뒤 섹션에서 볼 것

- 식별 문자가 구분 단서라서, 입력 크기를 너무 줄이면 글자가 뭉개질 수 있다. 5. bbox 분포에서 bbox 크기와 입력 크기를 볼 때 함께 본다. JSON의 알약 크기(mm)도 bbox 크기와 비교할 수 있다.
- 같은 조합의 사진끼리 얼마나 비슷한지는 6. 조합과 촬영 각도에서 본다. 1차 학습에 쓴 train/val 분할은 8-2에 정리한다.
- test에만 있는 클래스는 7. test 데이터에서 본다.

## 5. bbox 분포

bbox의 크기와 위치를 보고, 모델의 입력 크기를 정할 때 참고할 것을 찾는다.

| 단계 | 하는 일 | 이유 |
|---|---|---|
| 5-1 | bbox 크기 | 물체가 사진에서 얼마나 큰지 알아야 입력 크기와 anchor(모델이 미리 정해 둔 기준 박스 크기)를 판단할 수 있다. |
| 5-2 | 알약 크기(mm)와 bbox 크기 비교 | 같은 거리에서 찍었다면 1mm당 픽셀 수가 거의 일정해야 한다. 크게 벗어나는 bbox는 크기 오류이거나 촬영 조건이 다른 후보다. |
| 5-3 | bbox 위치 | 알약이 사진 전체에 퍼져 있는지 일부에 몰려 있는지 보면, 자르기(crop)나 이동 증강을 정할 때 참고가 된다. |
| 5-4 | 입력 크기에 주는 영향 | 4-5에서 식별 문자가 주요 구분 단서였다. 입력을 너무 줄이면 글자가 뭉개질 수 있다. |

- 이미지 밖 bbox 1개(3-2)는 크기 계산에는 넣고, 위치 계산에서는 뺀다.
- 면적은 JSON의 `area` 대신 너비 × 높이로 다시 계산한다. 3-6에서 `area`가 틀린 bbox가 있었다.

### 5-1. bbox 크기

- bbox마다 너비, 높이, 면적, 긴 변, 짧은 변, 긴 변 ÷ 짧은 변을 계산한다. 긴 변 ÷ 짧은 변이 1에 가까우면 정사각형에 가까운 bbox다.
- 이미지 크기에 대한 비율(bbox 너비 ÷ 이미지 너비, bbox 높이 ÷ 이미지 높이)도 계산한다. 입력 크기를 바꿔도 이 비율은 그대로라서, 줄였을 때 bbox가 몇 픽셀이 되는지 계산할 수 있다.
- COCO 방식의 크기 구분으로도 센다. COCO 평가는 면적이 32² 미만이면 small, 32²~96²이면 medium, 96² 이상이면 large로 나눠서 따로 점수를 낸다. 작은 물체가 많으면 작은 물체를 잘 찾는 설정이 필요하다.
- 너비, 높이, 면적, 긴 변 ÷ 짧은 변을 히스토그램으로 그린다.

In [ ]:
size_df = ann_df[["file_name", "category_id", "width", "height", "bbox_w", "bbox_h"]].copy()
size_df["면적"] = size_df["bbox_w"] * size_df["bbox_h"]  # JSON의 area는 3-6에서 틀린 값이 있어서 다시 계산한다
size_df["긴 변"] = size_df[["bbox_w", "bbox_h"]].max(axis=1)
size_df["짧은 변"] = size_df[["bbox_w", "bbox_h"]].min(axis=1)
size_df["긴 변 ÷ 짧은 변"] = size_df["긴 변"] / size_df["짧은 변"]
size_df["너비 비율"] = size_df["bbox_w"] / size_df["width"]
size_df["높이 비율"] = size_df["bbox_h"] / size_df["height"]

SIZE_COLUMNS = ["bbox_w", "bbox_h", "면적", "긴 변", "짧은 변", "긴 변 ÷ 짧은 변", "너비 비율", "높이 비율"]
print(f"bbox {len(size_df)}개")
display(size_df[SIZE_COLUMNS].describe().round(3))

coco_size = pd.cut(size_df["면적"], bins=[0, 32**2, 96**2, np.inf], right=False,
                   labels=["small (32² 미만)", "medium (32²~96²)", "large (96² 이상)"])
display(coco_size.value_counts(sort=False).rename_axis("COCO 크기 구분").reset_index(name="bbox 수"))

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), layout="constrained")
for ax, col, unit in zip(axes, ["bbox_w", "bbox_h", "면적", "긴 변 ÷ 짧은 변"], ["픽셀", "픽셀", "픽셀²", "배"]):
    ax.hist(size_df[col], bins=30, color="#2a78d6", edgecolor="white", linewidth=0.5)
    ax.axvline(size_df[col].median(), color="#555555", linewidth=1, linestyle="--")
    ax.set_title(f"{col} (점선: 중앙값 {size_df[col].median():.4g})", fontsize=10)
    ax.set_xlabel(unit)
axes[0].set_ylabel("bbox 수")
plt.show()

#### 5-1 결과

- 763개 모두 COCO 기준 large(면적 96² 이상)다. 가장 작은 bbox도 면적이 23250(약 152²)이다. COCO 방식으로 평가하면 small, medium 구간에는 물체가 없다.
- 너비는 125~529픽셀(중앙값 230), 높이는 126~664픽셀(중앙값 227)이다. 짧은 변은 가장 작아도 125픽셀이다.
- 이미지 크기에 대한 비율은 너비 0.128~0.542(중앙값 0.236), 높이 0.098~0.519(중앙값 0.177)다. 알약 하나가 보통 이미지 너비의 4분의 1쯤을 차지한다.
- 긴 변 ÷ 짧은 변은 1.0~2.53이고 중앙값은 1.076이다. 히스토그램에서 1.0 근처 막대 하나에 약 360개가 몰려 있다. 정사각형에 가까운 bbox가 절반 가까이 된다.
- 너비와 높이 히스토그램은 185~195픽셀 근처 막대가 가장 높고, 큰 쪽으로 길게 퍼진다.

### 5-2. 알약 크기(mm)와 bbox 크기 비교

- JSON의 `leng_long`(장축, mm)과 bbox의 긴 변(픽셀)을 비교한다. 4-4에서 같은 클래스의 JSON끼리 장축 값이 같았다. 16548은 장축이 0이라 뺀다.
- bbox마다 1mm당 픽셀 수(긴 변 ÷ 장축)를 계산한다. 같은 거리에서 찍었다면 이 값이 거의 일정해야 한다.
- 이 값은 원래도 조금 흩어진다.
    - bbox는 가로세로 축에 맞춘 사각형이라, 길쭉한 알약이 비스듬히 놓이면 bbox의 긴 변이 장축보다 짧아진다(45°로 놓이면 대략 (장축 + 단축) ÷ √2).
    - 알약의 두께와 카메라 각도(70°, 75°, 90°)에 따라 사진에 보이는 크기가 달라진다.
- 그래서 원형 알약(장축과 단축이 같음)과 그 외 알약을 나눠 색을 다르게 그린다. 원형은 놓인 방향의 영향을 받지 않는다.
- 많이 벗어난 값은 IQR 규칙으로 찾는다. IQR은 사분위 범위(75% 값 − 25% 값)이고, 25% 값 − 1.5 × IQR보다 작거나 75% 값 + 1.5 × IQR보다 크면 벗어난 값으로 본다.
- 벗어난 bbox는 사진에 파란색으로 그려서, bbox가 알약을 제대로 감싸는지 본다.

In [ ]:
mm_df = size_df.join(shape_df[["drug_shape", "leng_long", "leng_short"]], on="category_id")
mm_df = mm_df[mm_df["leng_long"] > 0].copy()  # 16548은 장축이 0이라 뺀다(4-4)
mm_df["px/mm"] = mm_df["긴 변"] / mm_df["leng_long"]
mm_df["원형"] = mm_df["leng_long"] == mm_df["leng_short"]

q1, q3 = mm_df["px/mm"].quantile([0.25, 0.75])
iqr_low, iqr_high = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
mm_out = mm_df[(mm_df["px/mm"] < iqr_low) | (mm_df["px/mm"] > iqr_high)].sort_values("px/mm")

print(f"비교한 bbox: {len(mm_df)}개 (원형 {int(mm_df['원형'].sum())}개, 그 외 {int((~mm_df['원형']).sum())}개)")
display(mm_df.groupby("원형")["px/mm"].describe().round(2).rename(index={True: "원형", False: "그 외"}))
print(f"IQR 범위: {iqr_low:.2f} ~ {iqr_high:.2f} px/mm, 범위 밖 bbox: {len(mm_out)}개")
if len(mm_out):
    display(mm_out.join(class_df["dl_name"], on="category_id")[
        ["file_name", "category_id", "dl_name", "drug_shape", "leng_long", "leng_short", "bbox_w", "bbox_h", "px/mm"]].round(2))

fig, ax = plt.subplots(figsize=(7, 5.5), layout="constrained")
for is_round, label, color in [(True, "원형", "tab:orange"), (False, "그 외", "tab:blue")]:
    part = mm_df[mm_df["원형"] == is_round]
    ax.scatter(part["leng_long"], part["긴 변"], s=14, alpha=0.6, color=color, label=label)
if len(mm_out):
    ax.scatter(mm_out["leng_long"], mm_out["긴 변"], s=70, facecolors="none", edgecolors="black", linewidths=1, label="IQR 범위 밖")
xs = np.array([0, mm_df["leng_long"].max() * 1.05])
ax.plot(xs, xs * mm_df["px/mm"].median(), color="#555555", linewidth=1, linestyle="--", label=f"중앙값 {mm_df['px/mm'].median():.1f} px/mm")
ax.set_xlabel("JSON의 장축 (mm)")
ax.set_ylabel("bbox의 긴 변 (픽셀)")
ax.set_title("알약 크기(mm)와 bbox 크기")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.show()

In [ ]:
MAX_SHOW = 12  # 범위 밖 bbox가 많으면 중앙값에서 많이 벗어난 순으로 이만큼만 그린다


def boxes_with_size_check(file_name, target_idx):
    """이미지 하나의 bbox 목록. 범위 밖 bbox만 파란색으로 그린다. 이미지 밖 bbox(3-2)는 그리지 않는다."""
    rows = crop_df[crop_df["file_name"] == file_name]
    return [(r.bbox_x, r.bbox_y, r.bbox_w, r.bbox_h, str(r.category_id), "#2a78d6" if r.Index == target_idx else "#e5484d")
            for r in rows.itertuples()]


ratio = np.log(mm_out["px/mm"] / mm_df["px/mm"].median()).abs()  # 중앙값보다 크든 작든 같은 기준으로 비교하려고 로그를 쓴다
show_out = mm_out.loc[ratio.sort_values(ascending=False).index[:MAX_SHOW]]
print(f"그릴 bbox: {len(show_out)}개 / 범위 밖 {len(mm_out)}개")

if len(show_out):
    show_rows = list(show_out.itertuples())  # px/mm 열은 이름에 /가 있어서 show_out.loc으로 꺼낸다
    show_thumbs(
        paths=[path_by_name[r.file_name] for r in show_rows],
        captions=[f"{r.file_name.split('_')[0]}\n각도 {r.file_name.split('_')[5]}°, {r.category_id}: {show_out.loc[r.Index, 'px/mm']:.2f} px/mm"
                  for r in show_rows],
        boxes=[boxes_with_size_check(r.file_name, r.Index) for r in show_rows],
        title="1mm당 픽셀 수가 IQR 범위 밖인 bbox (파란색)",
        ncols=4,
    )

#### 5-2 결과

- 비교한 bbox는 745개다(16548의 18개 제외). 원형 329개, 그 외 416개다.
- 1mm당 픽셀 수
    - 원형: 중앙값 26.55, 표준편차 0.81, 범위 22.47~29.24
    - 그 외: 중앙값 23.71, 표준편차 1.64, 범위 16.85~28.23
- 원형 알약은 값이 좁게 모여 있다. 사진마다 촬영 거리가 거의 같다는 뜻이다. 그 외 알약은 값이 더 낮고 더 흩어져 있다. 비스듬히 놓이면 bbox의 긴 변이 장축보다 짧아진다는 설명과 맞는다.
- 5-1에서 너비와 높이 히스토그램의 가장 높은 막대가 185~195픽셀 근처였다. bbox가 가장 많은 3351(4-2, 153개)은 장축 7.01mm의 원형 알약(4-4)이라, 원형 중앙값 26.55 px/mm로 계산하면 약 186픽셀이다. 이 막대는 3351의 bbox가 많아서 생긴 것으로 보인다.
- 산점도에서 점들이 장축에 비례하는 점선 근처에 모여 있다.
- IQR 범위(19.21~30.93 px/mm) 밖의 bbox는 2개이고, 둘 다 3번에서 이미 찾은 오류다.

| 사진 | 클래스 | px/mm | 3번에서 찾은 오류 | 오류를 고친다고 보면 |
|---|---|---|---|---|
| `K-003351-019232-029667` 70° | 19232 | 16.85 | bbox 높이 241 → 341 후보(3-5, 3-6) | 341 ÷ 14.3mm ≈ 23.8, 범위 안 |
| `K-003483-025367-027733-035206` 75° | 35206 | 18.88 | 27733과 클래스가 바뀜(3-5) | 이 bbox는 27733 알약(장축 14.0mm)을 감싸므로 321 ÷ 14.0 ≈ 22.9, 범위 안 |

- 클래스가 바뀐 다른 쪽인 27733 bbox(긴 변 380)는 27733의 장축 14.0mm로 나누면 27.1이라 범위 안이다. 그래서 이 방법으로는 잡히지 않았다.
- 이 방법으로 새로 찾은 오류는 없다. 다만 범위가 넓어서(19.21~30.93 px/mm), 전체 중앙값(25.3 px/mm)에서 약 20% 넘게 벗어난 bbox만 잡힌다.
- 각도(70°, 75°, 90°)별로 1mm당 픽셀 수가 다른지는 따로 보지 않았다.

### 5-3. bbox 위치

- 위치는 이미지 밖 bbox(3-2)를 뺀 762개로 본다.
- bbox 중심이 이미지의 어디에 있는지를 이미지 크기에 대한 비율(0~1)로 계산한다. 0.5면 가운데다.
- bbox에서 이미지 네 변까지의 여백(픽셀)을 계산한다. 여백이 작은 bbox가 많으면, 자르기 증강에서 알약이 잘리기 쉽다.
- 그림 두 개를 그린다.
    - bbox 중심점: 사진 크기의 틀 안에 중심점을 찍는다.
    - bbox가 덮은 횟수: 이미지를 8 × 8 픽셀 칸으로 나누고, 칸마다 몇 개의 bbox가 덮는지 센다. 진할수록 알약이 자주 놓인 곳이다.

In [ ]:
pos_df = ann_df.loc[crop_df.index, ["file_name", "category_id", "width", "height", "bbox_x", "bbox_y", "bbox_w", "bbox_h"]].copy()
pos_df["중심 x 비율"] = (pos_df["bbox_x"] + pos_df["bbox_w"] / 2) / pos_df["width"]
pos_df["중심 y 비율"] = (pos_df["bbox_y"] + pos_df["bbox_h"] / 2) / pos_df["height"]
pos_df["왼쪽 여백"] = pos_df["bbox_x"]
pos_df["위쪽 여백"] = pos_df["bbox_y"]
pos_df["오른쪽 여백"] = pos_df["width"] - (pos_df["bbox_x"] + pos_df["bbox_w"])
pos_df["아래쪽 여백"] = pos_df["height"] - (pos_df["bbox_y"] + pos_df["bbox_h"])
MARGIN_COLUMNS = ["왼쪽 여백", "위쪽 여백", "오른쪽 여백", "아래쪽 여백"]

print(f"bbox {len(pos_df)}개")
display(pos_df[["중심 x 비율", "중심 y 비율", *MARGIN_COLUMNS]].describe().round(2))

EDGE = 20  # 이 픽셀 수보다 가장자리에 가까우면 '가장자리에 붙은 bbox'로 센다
near_edge = (pos_df[MARGIN_COLUMNS] < EDGE).any(axis=1)
print(f"네 변 중 하나라도 여백이 {EDGE}픽셀 미만인 bbox: {int(near_edge.sum())}개 / {len(pos_df)}개")
display((pos_df[MARGIN_COLUMNS] < EDGE).sum().rename_axis("변").reset_index(name=f"여백 {EDGE}픽셀 미만 bbox 수"))

assert pos_df["width"].nunique() == 1 and pos_df["height"].nunique() == 1, "이미지 크기가 한 가지가 아닙니다. 2-2를 확인하세요."
W, H = int(pos_df["width"].iloc[0]), int(pos_df["height"].iloc[0])
GRID = 8  # 덮은 횟수를 셀 칸의 크기(픽셀)
cover = np.zeros((H // GRID + 1, W // GRID + 1))
for r in pos_df.itertuples():
    cover[r.bbox_y // GRID:(r.bbox_y + r.bbox_h) // GRID + 1, r.bbox_x // GRID:(r.bbox_x + r.bbox_w) // GRID + 1] += 1

fig, axes = plt.subplots(1, 2, figsize=(10, 6.5), layout="constrained")
axes[0].scatter(pos_df["bbox_x"] + pos_df["bbox_w"] / 2, pos_df["bbox_y"] + pos_df["bbox_h"] / 2, s=10, alpha=0.5, color="#2a78d6")
axes[0].set_xlim(0, W)
axes[0].set_ylim(H, 0)  # 이미지 좌표처럼 위쪽이 0
axes[0].set_aspect("equal")
axes[0].set_title("bbox 중심점")
im = axes[1].imshow(cover, cmap="Blues", extent=(0, W, H, 0))
axes[1].set_title(f"bbox가 덮은 횟수 ({GRID} × {GRID} 픽셀 칸)")
fig.colorbar(im, ax=axes[1], label="덮은 bbox 수", shrink=0.8)
for ax in axes:
    ax.set_xlabel("x (픽셀)")
    ax.set_ylabel("y (픽셀)")
plt.show()

#### 5-3 결과

- bbox 중심은 이미지 너비의 0.15~0.82, 높이의 0.16~0.80 사이에 있다. 중심이 가장자리 가까이에 있는 bbox는 없다.
- 여백이 20픽셀 미만인 bbox는 16개다(왼쪽 11개, 오른쪽 4개, 아래쪽 1개, 위쪽 0개). 여백의 최솟값은 왼쪽 2, 오른쪽 1, 아래쪽 11, 위쪽 25픽셀이다.

그림에서 보이는 것

- bbox 중심점이 6곳에 몰려 있다. 위쪽에 3곳(왼쪽, 가운데, 오른쪽), 아래쪽에 3곳이다. 위쪽 가운데는 양옆보다 조금 높고, 아래쪽 가운데는 양옆보다 조금 낮다.
- 이미지 가운데의 가로 띠(y 약 500~700)와 네 모서리에는 중심점이 거의 없다.
- 덮은 횟수 지도에서는 아래쪽 왼쪽과 오른쪽이 가장 진하다.
- 알약을 정해진 몇 자리에 놓고 찍은 것으로 보인다. train에서는 알약 위치가 다양하지 않다.

test 사진은 라벨이 없어서, 알약이 같은 자리에 놓였는지는 이 EDA에서 확인하지 않았다.

### 5-4. 입력 크기에 주는 영향

- 검출 모델은 보통 이미지의 긴 변을 정해진 입력 크기에 맞춰 줄인다(예: YOLO의 기본 입력 크기 640). 원본은 976 × 1280이라 긴 변이 1280이다.
- 입력 크기 후보 1280, 960, 640, 416으로 줄였을 때, 5-1의 bbox 짧은 변과 긴 변이 몇 픽셀이 되는지 계산한다. 1280은 원본 그대로다.
- 숫자만으로는 글자가 읽히는지 알 수 없어서, 알약 세 개를 같은 비율로 줄여 본다.
    - 가장 작은 bbox(면적 기준): 가장 크게 줄어드는 경우
    - 25469(아모잘탄정): 4-4 그림에서 `Amosartan 5/100`처럼 작은 글자가 인쇄돼 있었다.
    - 20014(마도파정): 4-4 그림에서 가장자리를 따라 작은 글자와 십자 분할선이 새겨져 있었다.
- 줄인 이미지에서 bbox 영역을 잘라, 모두 같은 크기로 확대해서 보여 준다. 확대할 때 픽셀을 섞지 않게(nearest) 해서, 줄인 이미지에 실제로 남은 픽셀이 그대로 보이게 한다.
- 이미지는 PIL의 BILINEAR로 줄인다. 실제 학습 코드가 쓰는 줄이는 방식과는 조금 다를 수 있다.

In [ ]:
INPUT_SIZES = [1280, 960, 640, 416]
long_side = max(W, H)  # 5-3에서 구한 이미지 크기

input_rows = []
for size in INPUT_SIZES:
    scale = size / long_side
    input_rows.append({
        "입력 크기": size,
        "줄이는 비율": round(scale, 3),
        "줄인 이미지 (너비 × 높이)": f"{round(W * scale)} × {round(H * scale)}",
        "짧은 변 최소": round(size_df["짧은 변"].min() * scale, 1),
        "짧은 변 중앙값": round(size_df["짧은 변"].median() * scale, 1),
        "긴 변 최소": round(size_df["긴 변"].min() * scale, 1),
        "긴 변 중앙값": round(size_df["긴 변"].median() * scale, 1),
    })
display(pd.DataFrame(input_rows))

In [ ]:
smallest_idx = size_df.loc[crop_df.index, "면적"].idxmin()  # 이미지 밖 bbox(3-2)는 자를 수 없어서 뺀다
EXAMPLES = [
    (smallest_idx, "가장 작은 bbox"),
    (class_cells(25469)[0], "작은 인쇄 글자"),
    (class_cells(20014)[0], "가장자리의 작은 각인"),
]
SHOW_PX = 200  # 확대해서 보여 줄 크기(픽셀)

fig, axes = plt.subplots(len(EXAMPLES), len(INPUT_SIZES), figsize=(3 * len(INPUT_SIZES), 3.3 * len(EXAMPLES)), layout="constrained", squeeze=False)
for row_axes, (idx, desc) in zip(axes, EXAMPLES):
    r = ann_df.loc[idx]
    with Image.open(path_by_name[r["file_name"]]) as im:
        img = im.convert("RGB")
    for ax, size in zip(row_axes, INPUT_SIZES):
        scale = size / long_side
        small = img.resize((round(W * scale), round(H * scale)), Image.BILINEAR)
        box = (round(r["bbox_x"] * scale), round(r["bbox_y"] * scale),
               round((r["bbox_x"] + r["bbox_w"]) * scale), round((r["bbox_y"] + r["bbox_h"]) * scale))
        crop = small.crop(box)
        ax.imshow(crop.resize((SHOW_PX, round(SHOW_PX * crop.height / crop.width)), Image.NEAREST))
        ax.set_title(f"입력 {size}: bbox {crop.width} × {crop.height}", fontsize=9)
        ax.axis("off")
    row_axes[0].set_title(f"{r['category_id']} ({desc})\n입력 {INPUT_SIZES[0]}: bbox {r['bbox_w']} × {r['bbox_h']}", fontsize=9)
fig.suptitle("입력 크기별로 줄였을 때의 bbox 영역 (같은 크기로 확대)", fontsize=11)
plt.show()

#### 5-4 결과

| 입력 크기 | 줄이는 비율 | 짧은 변 최소 | 짧은 변 중앙값 |
|---|---|---|---|
| 1280 (원본) | 1.000 | 125.0 | 214.0 |
| 960 | 0.750 | 93.8 | 160.5 |
| 640 | 0.500 | 62.5 | 107.0 |
| 416 | 0.325 | 40.6 | 69.6 |

- 416으로 줄여도 가장 작은 bbox의 짧은 변이 40픽셀이 넘는다. 물체 크기만 보면 네 후보 모두 너무 작아지는 bbox는 없다.
- 그림의 가장 작은 bbox는 19607(155 × 150)이다. 5-1에서 면적이 가장 작았던 bbox(23250)다.

그림에서 보이는 것

| 알약 | 1280, 960 | 640 | 416 |
|---|---|---|---|
| 19607 (FZ12 각인) | 각인이 또렷하다. | 각인이 흐려지지만 획은 보인다. | 픽셀이 굵게 보이고, 획이 희미해서 글자를 알아보기 어렵다. |
| 25469 (Amosartan 5/100 인쇄) | 두 줄 모두 읽힌다. | Amosartan은 읽히고, 5/100은 흐려진다. | Amosartan은 대략 보이지만, 5/100은 알아보기 어렵다. |
| 20014 (십자 분할선, 가장자리 글자) | 분할선과 글자가 또렷하다. | 분할선과 글자가 보인다. | 분할선은 뚜렷하고, 가장자리 글자는 흐려진다. |

- 640까지는 세 알약 모두 각인이나 인쇄 글자가 보인다. 416에서는 작은 각인과 작은 글자가 흐려진다.
- 4-4에서 49개 클래스가 모양과 색이 같은 다른 클래스와 묶였다. 이 클래스들은 식별 문자가 주요 구분 단서라서, 입력을 너무 줄이면 구분이 어려워질 수 있다.
- 입력 크기를 키우면 학습에 드는 메모리와 시간이 늘어난다.
- 세 알약만 본 결과라서, 다른 알약의 글자가 어떻게 되는지는 이 그림만으로 알 수 없다.

### 5-5. 정리

#### 확인한 것

- 크기
    - bbox 763개 모두 COCO 기준 large다. 짧은 변은 가장 작아도 125픽셀이다.
    - 알약 하나가 보통 이미지 너비의 4분의 1쯤을 차지하고, 정사각형에 가까운 bbox가 절반 가까이 된다.
- 알약 크기(mm)와 bbox 크기
    - 원형 알약은 1mm당 약 26.5픽셀로 거의 일정하다. 사진마다 촬영 거리가 거의 같다.
    - 범위 밖 bbox 2개는 3번에서 찾은 오류였고, 새로 찾은 오류는 없다.
- 위치
    - 그림에서 bbox 중심이 6곳에 몰려 있었다. train에서는 알약 위치가 다양하지 않다.
    - 가장자리 여백이 20픽셀 미만인 bbox는 16개다.
- 입력 크기
    - 640으로 줄이면 가장 작은 bbox의 짧은 변이 62.5픽셀이다. 그림에서 본 세 알약은 640까지 각인이 보였다.
    - 416에서는 작은 각인과 작은 글자가 흐려졌다.

#### 정할 것 (이 EDA에서는 정하지 않는다)

- 입력 크기: 640과 더 큰 크기(960, 1280) 중에서, 각인을 남기는 것과 학습 비용 사이에서 고른다.
- 위치와 방향 증강
    - train에서는 알약 위치가 몇 자리에 몰려 있다. 4-4 그림에서는 알약이 놓인 방향이 여러 가지였다.
    - 이동, 회전, 뒤집기 증강을 쓸지 정한다.
    - 자르기 증강을 쓰면 가장자리에 붙은 bbox 16개가 잘릴 수 있다.

#### 뒤 섹션에서 볼 것

- 같은 조합 사진끼리 알약 위치와 방향이 어떻게 다른지는 6. 조합과 촬영 각도에서 본다.

#### 이 EDA에서 확인하지 않은 것

- test 사진은 라벨이 없어서, 알약이 같은 자리에 비슷한 크기로 찍혔는지는 확인하지 않았다.

## 6. 조합과 촬영 각도

같은 조합 이름의 사진은 같은 알약들을 각도와 앞뒷면만 바꿔 찍은 것이다(1-8). 같은 조합의 사진이 몇 장씩 있고 서로 얼마나 비슷한지 보고, train/val을 나눌 단위를 정하는 근거로 쓴다.

| 단계 | 하는 일 | 이유 |
|---|---|---|
| 6-1 | 조합당 사진 수와 각도 구성 | 같은 조합의 사진이 몇 장씩 묶여 있는지 알아야 분할 단위를 정할 수 있다. |
| 6-2 | 같은 조합 사진 비교 | 같은 조합의 사진끼리 알약 위치와 방향이 얼마나 같은지 눈으로 본다. |
| 6-3 | 같은 조합 사진의 bbox 비교 | 6-2에서 보인 것을 숫자로 확인한다. 비교 방법은 6-2 결과를 보고 정한다. |
| 6-4 | 분할 단위에 주는 의미 | 사진 단위로 나눌 때와 조합 단위로 나눌 때의 차이를 정리한다. |

### 6-1. 조합당 사진 수와 각도 구성

- 2-3의 이미지 표(`image_df`)를 쓴다. 이미지마다 조합 이름, 각도(`camera_la`), 앞뒷면(`drug_dir`)이 있다.
- 조합마다 사진 수, 각도 구성(어떤 각도의 사진이 있는지), 뒷면 사진 수, 조합 이름의 알약 코드 수를 표(`combo_df`)로 만든다.
- 다음을 센다.
    - 사진 수별 조합 수
    - 각도 구성별 조합 수
    - 같은 조합에 같은 각도의 사진이 2장 이상인 경우
    - 알약 코드 수(3개, 4개)와 사진 수의 관계
- 뒷면 사진 4장(1-8)은 어느 조합의 어느 각도인지, 같은 조합에 다른 사진이 있는지 본다.

In [ ]:
photo_df = image_df[["file_name", "combo", "camera_la", "drug_dir"]].copy()
photo_df["bbox 수"] = photo_df["file_name"].map(ann_df.groupby("file_name").size())

combo_df = photo_df.groupby("combo").agg(**{
    "사진 수": ("file_name", "size"),
    "각도 구성": ("camera_la", lambda s: ", ".join(str(a) for a in sorted(s))),
    "뒷면 사진 수": ("drug_dir", lambda s: int((s == "뒷면").sum())),
})
combo_df["알약 코드 수"] = [len(pill_codes(c)) for c in combo_df.index]  # 1-3의 함수. 조합 이름도 그대로 넣을 수 있다

print(f"조합: {len(combo_df)}개, 사진: {len(photo_df)}장")
print()
print("사진 수별 조합 수")
display(combo_df["사진 수"].value_counts().sort_index().rename_axis("사진 수").reset_index(name="조합 수"))
print("각도 구성별 조합 수")
display(combo_df["각도 구성"].value_counts().rename_axis("각도 구성").reset_index(name="조합 수"))

same_angle = photo_df.groupby(["combo", "camera_la"]).size()
print(f"같은 조합에 같은 각도의 사진이 2장 이상인 경우: {int((same_angle > 1).sum())}건")
if (same_angle > 1).any():
    display(same_angle[same_angle > 1].rename("사진 수").reset_index())

print()
print("알약 코드 수와 사진 수 (칸의 값: 조합 수)")
display(pd.crosstab(combo_df["알약 코드 수"], combo_df["사진 수"], margins=True, margins_name="합계"))

back_photos = photo_df[photo_df["drug_dir"] == "뒷면"]
print(f"뒷면 사진: {len(back_photos)}장")
if len(back_photos):
    display(back_photos.join(combo_df[["사진 수", "각도 구성", "뒷면 사진 수"]], on="combo")
            [["file_name", "camera_la", "사진 수", "각도 구성", "뒷면 사진 수"]])

#### 6-1 결과

- 조합 114개에 사진 232장이다. 조합당 사진은 1장인 조합이 36개, 2장이 38개, 3장이 40개다.
- 같은 조합에 같은 각도의 사진이 2장 이상인 경우는 없다. 조합 하나에 각도마다 사진이 한 장씩이다.

| 각도 구성 | 조합 수 |
|---|---|
| 70, 75, 90 | 40 |
| 두 각도 (70·90 15개, 75·90 14개, 70·75 9개) | 38 |
| 한 각도 (90 16개, 75 12개, 70 8개) | 36 |

- 알약 코드가 4개인 조합 25개는 모두 사진이 3장이다. 알약 코드가 3개인 조합 89개는 1장 36개, 2장 38개, 3장 15개다.
- 뒷면 사진 4장은 조합 2개에 2장씩 있다(`K-003351-019232-029667` 70°·90°, `K-003351-020014-036637` 75°·90°). 두 조합은 사진이 모두 뒷면이고, 같은 조합의 앞면 사진은 없다.
- 알약 코드가 3개인 조합은 74개가 사진이 3장보다 적다. 빠진 각도의 사진이 다른 곳(예: test)에 있는지는 이 표만으로 알 수 없고, 이 EDA에서는 확인하지 않았다.

### 6-2. 같은 조합 사진 비교

- 각도가 70°, 75°, 90° 세 개이고 뒷면 사진이 없는 조합 중 4개를 무작위로 고른다. seed를 고정해서 다시 실행해도 같은 조합이 나온다.
- 조합마다 그림 하나에 세 각도의 사진을 한 줄로 나란히 그리고, bbox와 클래스 번호를 함께 그린다. 같은 클래스의 알약이 각도마다 어디에 있는지 보기 위해서다.
- 6-1에서 뒷면 사진이 있는 조합은 그 조합의 사진을 모두 따로 그린다. 앞면 사진과 무엇이 다른지 보기 위해서다.

In [ ]:
N_COMBO_SAMPLES = 4

three_angle = sorted(combo_df.index[(combo_df["각도 구성"] == "70, 75, 90") & (combo_df["뒷면 사진 수"] == 0)])
combo_rng = np.random.default_rng(SEED)  # 셀을 다시 실행해도 같은 조합이 뽑히도록 셀 안에서 새로 만든다
picked_combos = [three_angle[i] for i in sorted(combo_rng.choice(len(three_angle), N_COMBO_SAMPLES, replace=False))]
print(f"각도가 셋이고 뒷면 사진이 없는 조합: {len(three_angle)}개 중 {N_COMBO_SAMPLES}개")


def show_combo(combo, title_prefix):
    """한 조합의 사진을 각도 순서로 한 줄에 그린다. 3-1의 show_thumbs와 boxes_for를 쓴다."""
    rows = photo_df[photo_df["combo"] == combo].sort_values(["camera_la", "drug_dir"])
    show_thumbs(
        paths=[path_by_name[n] for n in rows["file_name"]],
        captions=[f"각도 {r.camera_la}°, {r.drug_dir}" for r in rows.itertuples()],
        boxes=[boxes_for(n) for n in rows["file_name"]],
        title=f"{title_prefix}: {combo}",
        ncols=len(rows),
    )


for combo in picked_combos:
    show_combo(combo, "세 각도 사진")

back_combos = list(dict.fromkeys(back_photos["combo"]))  # 순서를 지키며 중복 제거
for combo in back_combos:
    show_combo(combo, "뒷면 사진이 있는 조합")

#### 6-2 결과

고른 조합은 `K-001900-016548-024850-029345`, `K-003351-018147-029667`, `K-003483-016232-022347-025469`, `K-003483-020238-027733-028763`이다.

그림에서 보이는 것

- 75°와 90° 사진은 알약이 같은 자리에 같은 방향으로 놓여 있다. 위치가 조금 어긋난 정도라 거의 같은 사진으로 보인다.
- 70° 사진은 알약의 자리가 75°, 90°와 다르다.
    - 75°, 90°에서 왼쪽 위에 있던 알약이 70°에서는 오른쪽 아래에 있고, 오른쪽 위에 있던 알약은 왼쪽 아래에 있다. 위아래와 좌우가 모두 바뀌어 있다.
    - 각인 글자도 거꾸로 보인다(예: `K-003351-018147-029667`의 3351 HT, 29667 203).
    - 사진을 180° 돌린 것과 같은 배치로 보인다.
- 뒷면 사진이 있는 두 조합도 같다. `K-003351-019232-029667`의 70°와 90°는 180° 돌린 배치로 보이고, `K-003351-020014-036637`의 75°와 90°는 같은 배치로 보인다.
- 뒷면 사진에서도 29667은 Kowa 203(`print_front`)이, 3351은 HT(`print_back`)가 보인다. 3-5에서처럼 `drug_dir`이 뒷면이어도 알약마다 보이는 면은 다르다.

같은 조합에서 75°와 90°는 거의 같은 사진이고, 70°는 180° 돌린 사진으로 보인다. 6-3에서 이것을 bbox 좌표로 확인한다.

### 6-3. 같은 조합 사진의 bbox 비교

6-2에서 보인 두 가지(75°와 90°는 같은 배치, 70°는 180° 돌린 배치)를 bbox 좌표로 확인한다.

- 같은 조합에서 각도 쌍(70-75, 70-90, 75-90)마다, 같은 클래스의 bbox끼리 짝을 짓는다. 4-1에서 한 사진에 같은 클래스는 하나뿐이어서 짝이 하나로 정해진다.
- 짝마다 IoU를 두 가지로 계산한다. IoU 함수는 3-3에서 만든 것을 쓴다.
    - IoU 그대로: 두 bbox를 그대로 비교한다. 배치가 같으면 높다.
    - IoU 180° 회전: 앞 각도의 bbox를 이미지 가운데를 중심으로 180° 돌려서 비교한다. `[x, y, w, h]`를 돌리면 `[W − x − w, H − y − h, w, h]`가 된다(W, H는 이미지 너비와 높이). 180° 돌린 배치면 높다.
- 각도 쌍별로 두 IoU의 중앙값과, IoU가 0.5 이상인 비율을 센다. 히스토그램도 그린다.
- 이미지 밖 bbox(3-2)는 뺀다. 3-8의 다른 오류 bbox는 그대로 두어서, 낮은 IoU의 원인일 수 있다. 두 IoU 중 큰 값도 0.5 미만인 짝은 따로 표로 보여 준다.

In [ ]:
ANGLE_PAIRS = [(70, 75), (70, 90), (75, 90)]


def rotate180(bbox):
    """bbox [x, y, w, h]를 이미지 가운데를 중심으로 180° 돌린다. W, H는 5-3에서 구한 이미지 크기."""
    x, y, w, h = bbox
    return [W - x - w, H - y - h, w, h]


# (조합, 각도) → {category_id: bbox}. 이미지 밖 bbox(3-2)는 뺀다.
bbox_by_photo = {}
for r in ann_df.loc[crop_df.index].itertuples():
    bbox_by_photo.setdefault((r.combo, r.camera_la), {})[r.category_id] = [r.bbox_x, r.bbox_y, r.bbox_w, r.bbox_h]

match_rows = []
for combo in combo_df.index:
    for a1, a2 in ANGLE_PAIRS:
        b1, b2 = bbox_by_photo.get((combo, a1)), bbox_by_photo.get((combo, a2))
        if b1 is None or b2 is None:
            continue
        for cid in sorted(set(b1) & set(b2)):  # 두 사진에 모두 bbox가 있는 클래스
            match_rows.append({"combo": combo, "각도 쌍": f"{a1}-{a2}", "category_id": cid,
                               "IoU 그대로": iou(b1[cid], b2[cid]), "IoU 180° 회전": iou(rotate180(b1[cid]), b2[cid])})
match_df = pd.DataFrame(match_rows)
match_df["큰 IoU"] = match_df[["IoU 그대로", "IoU 180° 회전"]].max(axis=1)

print(f"짝지은 bbox: {len(match_df)}쌍 (조합 {match_df['combo'].nunique()}개)")
display(match_df.groupby("각도 쌍").agg(**{
    "bbox 쌍 수": ("category_id", "size"),
    "IoU 그대로 중앙값": ("IoU 그대로", "median"),
    "IoU 180° 회전 중앙값": ("IoU 180° 회전", "median"),
    "그대로 0.5 이상 비율": ("IoU 그대로", lambda s: (s >= 0.5).mean()),
    "180° 회전 0.5 이상 비율": ("IoU 180° 회전", lambda s: (s >= 0.5).mean()),
}).round(3))

fig, axes = plt.subplots(1, len(ANGLE_PAIRS), figsize=(15, 3.8), layout="constrained", sharey=True)
bins = np.linspace(0, 1, 21)
for ax, (a1, a2) in zip(axes, ANGLE_PAIRS):
    part = match_df[match_df["각도 쌍"] == f"{a1}-{a2}"]
    ax.hist(part["IoU 그대로"], bins=bins, alpha=0.6, color="tab:blue", label="그대로")
    ax.hist(part["IoU 180° 회전"], bins=bins, alpha=0.6, color="tab:orange", label="180° 회전")
    ax.set_title(f"{a1}° - {a2}° ({len(part)}쌍)", fontsize=10)
    ax.set_xlabel("IoU")
axes[0].set_ylabel("bbox 쌍 수")
axes[0].legend()
plt.show()

low_match = match_df[match_df["큰 IoU"] < 0.5].sort_values("큰 IoU")
print(f"두 IoU 모두 0.5 미만인 짝: {len(low_match)}쌍")
if len(low_match):
    display(low_match.round(3))

#### 6-3 결과

- 짝지은 bbox는 538쌍이고, 사진이 2장 이상인 조합 78개(6-1의 38 + 40)에서 나왔다.

| 각도 쌍 | bbox 쌍 | IoU 그대로 중앙값 | IoU 180° 회전 중앙값 | 그대로 0.5 이상 | 180° 회전 0.5 이상 |
|---|---|---|---|---|---|
| 70-75 | 169 | 0.000 | 0.690 | 1.2% | 72.2% |
| 70-90 | 185 | 0.000 | 0.763 | 0.0% | 98.9% |
| 75-90 | 184 | 0.526 | 0.000 | 56.5% | 0.5% |

- 70°가 들어간 쌍은 그대로 비교하면 IoU가 거의 모두 0이고, 70° bbox를 180° 돌리면 겹친다. 6-2에서 본 "70°는 180° 돌린 배치"가 전체 조합에서도 맞다.
- 75-90 쌍은 그대로 비교할 때 겹치고, 돌리면 겹치지 않는다. 75°와 90°는 같은 방향의 배치다.
- 다만 75-90의 IoU는 중앙값 0.526이고, 히스토그램에서 0.2~0.95로 넓게 퍼져 있다. 6-2 그림에서는 거의 같아 보였지만, bbox 기준으로는 위치가 꽤 어긋나는 짝이 많다.
- 70°를 돌렸을 때는 75°보다 90°와 더 잘 겹친다(중앙값 0.690과 0.763).
- 두 IoU가 모두 0.5 미만인 짝은 126쌍이다. 표에 보이는 가장 낮은 짝들은 3-8의 오류와 같은 사진, 같은 클래스다.
    - 16548 (`K-001900-016548-019607-033009`, 70-75와 70-90): 70°의 bbox가 다른 알약 위에 있음(3-3)
    - 20238 (`K-003351-020238-031863`, 70-90): 70°의 bbox가 다른 알약 위에 있음(3-3)
    - 3351 (`K-003351-029667-031863`, 70-75): 70°의 bbox가 다른 알약 위에 있음(3-3)
    - 3351 (`K-003351-020014-020238`, 75-90): 90°의 bbox가 빈 바닥에 있음(3-5)
- 표가 126행이라 가운데가 생략되어, 전체 목록은 아래 6-3 추가에서 다시 본다.

#### 6-3 추가. 위치가 어긋난 정도와 IoU가 아주 낮은 짝

- 각도 쌍마다 맞는 방향으로 맞춘다. 75-90은 그대로, 70°가 들어간 쌍은 70° bbox를 180° 돌린다. 이렇게 맞춘 IoU를 `맞춘 IoU`라고 한다.
- 맞춘 뒤 두 bbox의 중심이 몇 픽셀 어긋나는지(dx, dy, 거리)를 계산한다. 뒤 각도의 중심에서 앞 각도의 중심을 뺀 값이다. 어긋남이 짝마다 비슷한 방향이면 촬영 각도 때문에 생긴 차이이고, 제각각이면 알약을 다시 놓았거나 라벨 차이일 수 있다.
- 각도 쌍별로 dx, dy를 점으로 찍어 본다. 맞춘 IoU가 0.2 미만인 짝은 수백 픽셀 떨어져 있어서, 함께 그리면 나머지 점이 한곳에 뭉쳐 보인다. 그래서 그림에서는 빼고, 표로 따로 보여 준다.
- 맞춘 IoU가 0.2 미만인 짝은 위치 차이로 설명하기 어려워서 전체 목록을 보여 준다. 3-8의 오류와 비교한다.

In [ ]:
match_df["맞춘 IoU"] = np.where(match_df["각도 쌍"] == "75-90", match_df["IoU 그대로"], match_df["IoU 180° 회전"])

shift_rows = []
for m in match_df.to_dict("records"):  # 열 이름에 띄어쓰기가 있어서 dict로 꺼낸다
    a1, a2 = (int(a) for a in m["각도 쌍"].split("-"))
    b1 = bbox_by_photo[(m["combo"], a1)][m["category_id"]]
    b2 = bbox_by_photo[(m["combo"], a2)][m["category_id"]]
    if a1 == 70:
        b1 = rotate180(b1)
    dx = (b2[0] + b2[2] / 2) - (b1[0] + b1[2] / 2)  # 중심 x 차이
    dy = (b2[1] + b2[3] / 2) - (b1[1] + b1[3] / 2)  # 중심 y 차이
    shift_rows.append({"dx": dx, "dy": dy, "중심 거리": float(np.hypot(dx, dy))})
match_df[["dx", "dy", "중심 거리"]] = pd.DataFrame(shift_rows, index=match_df.index)

display(match_df.groupby("각도 쌍").agg(**{
    "bbox 쌍 수": ("category_id", "size"),
    "맞춘 IoU 중앙값": ("맞춘 IoU", "median"),
    "맞춘 IoU 0.5 이상 비율": ("맞춘 IoU", lambda s: (s >= 0.5).mean()),
    "dx 중앙값": ("dx", "median"),
    "dy 중앙값": ("dy", "median"),
    "중심 거리 중앙값": ("중심 거리", "median"),
    "중심 거리 90% 값": ("중심 거리", lambda s: s.quantile(0.9)),
}).round(3))

VERY_LOW = 0.2  # 맞춘 IoU가 이보다 낮은 짝은 위치 차이로 설명하기 어렵다고 본다
very_low = match_df[match_df["맞춘 IoU"] < VERY_LOW].sort_values("맞춘 IoU")
near = match_df.drop(very_low.index)  # 아주 낮은 짝은 수백 픽셀 떨어져 있어서, 함께 그리면 나머지 점이 한곳에 뭉쳐 보인다

fig, axes = plt.subplots(1, len(ANGLE_PAIRS), figsize=(15, 4.8), layout="constrained", sharex=True, sharey=True)
for ax, (a1, a2) in zip(axes, ANGLE_PAIRS):
    part = near[near["각도 쌍"] == f"{a1}-{a2}"]
    ax.scatter(part["dx"], part["dy"], s=12, alpha=0.6, color="#2a78d6")
    ax.axhline(0, color="#999999", linewidth=0.8)
    ax.axvline(0, color="#999999", linewidth=0.8)
    title = f"{a1}° (180° 회전) → {a2}°" if a1 == 70 else f"{a1}° → {a2}°"
    ax.set_title(f"{title}, {len(part)}쌍", fontsize=10)
    ax.set_xlabel("dx (픽셀)")
    ax.set_aspect("equal")
axes[0].set_ylabel("dy (픽셀)")
fig.suptitle(f"맞춘 뒤 bbox 중심이 어긋난 정도 (뒤 각도의 중심에서 앞 각도의 중심을 뺀 값, 맞춘 IoU {VERY_LOW} 미만인 짝 제외)", fontsize=11)
plt.show()

print(f"맞춘 IoU가 {VERY_LOW} 미만인 짝: {len(very_low)}쌍")
if len(very_low):
    with pd.option_context("display.max_rows", None):
        display(very_low[["combo", "각도 쌍", "category_id", "맞춘 IoU", "dx", "dy", "중심 거리"]].round(3))

#### 6-3 추가 결과

| 각도 쌍 | 맞춘 IoU 중앙값 | 맞춘 IoU 0.5 이상 | dx 중앙값 | dy 중앙값 | 중심 거리 중앙값 | 중심 거리 90% 값 |
|---|---|---|---|---|---|---|
| 70-75 (70° 회전) | 0.690 | 72.2% | −23.0 | −38.5 | 45.4 | 81.8 |
| 70-90 (70° 회전) | 0.763 | 98.9% | 8.0 | 13.5 | 30.1 | 42.9 |
| 75-90 | 0.526 | 56.5% | 33.5 | 41.0 | 52.4 | 100.0 |

- 70°를 돌려 90°와 맞추면 중심이 가장 적게 어긋난다. 중앙값 30픽셀이고, 90%가 43픽셀 안이다.
- 75°가 들어간 쌍은 더 많이 어긋난다. 어긋난 방향의 중앙값이 70→75는 (−23, −38.5), 75→90은 (33.5, 41)이다. 두 값 모두 75°의 bbox 중심이 다른 두 각도보다 왼쪽 위에 있다는 뜻이다.
- 75-90의 IoU 중앙값이 0.5 근처인 것은 이 차이로 설명된다. 5-1의 bbox 크기 정도(약 220 × 220)인 bbox가 (33, 41)만큼 밀리면 IoU는 약 0.53이다.
- 맞춘 IoU가 0.2 미만인 짝 9쌍은 모두 3-8의 오류다. 이 방법으로 새로 찾은 오류는 없다.
    - 16548 (`K-001900-016548-019607-033009`, 70-75와 70-90), 20238 (`K-003351-020238-031863`, 70-90), 3351 (`K-003351-029667-031863`, 70-75): 70°의 bbox가 다른 알약 위에 있음(3-3)
    - 27733, 35206 (`K-003483-025367-027733-035206`, 70-75와 75-90, 4쌍): 75°에서 두 클래스가 바뀜(3-5). 75°가 들어간 쌍만 맞지 않는다.
    - 3351 (`K-003351-020014-020238`, 75-90): 90°의 bbox가 빈 바닥에 있음(3-5)

그림에서 보이는 것 (맞춘 IoU 0.2 미만인 9쌍을 뺀 529쌍)

- 70→75는 점이 대부분 왼쪽 아래(dx 약 −55~10, dy 약 −95~10)에 모여 있다.
- 75→90은 점이 대부분 오른쪽 위에 있다. dx는 약 15~45로 비슷하고, dy는 약 5~105로 세로로 퍼져 있으며 몇 개의 무리로 나뉜다. 왼쪽 아래에도 점이 조금 있다.
- 70→90은 점이 원점 주변(dx 약 −45~35, dy 약 −80~45)에 퍼져 있다. 한쪽으로 몰리지 않았다.
- 75°가 들어간 두 쌍은 짝마다 비슷한 방향으로 어긋나 있어서, 알약을 다시 놓은 것보다는 75° 사진 전체가 한쪽으로 밀린 차이로 보인다. 75→90의 dy가 무리로 나뉘는 것은 사진 안의 위치에 따라 밀린 정도가 다르기 때문일 수 있다.

### 6-4. 분할 단위에 주는 의미

train과 val을 나누는 단위를 두 가지로 비교한다.

| 분할 단위 | 방법 | 장점 | 단점 |
|---|---|---|---|
| 사진 단위 | 사진 232장을 무작위로 나눈다. | 클래스가 train과 val에 고르게 들어가기 쉽다. | 같은 조합의 다른 각도 사진이 train과 val에 나뉠 수 있다. |
| 조합 단위 | 조합을 통째로 train이나 val에 넣는다. | 같은 조합의 사진이 양쪽에 나뉘지 않는다. | 조합이 1개뿐인 클래스 17개(4-3)는 한쪽에만 들어간다. |

- 6-3에서 같은 조합의 사진은 70°를 180° 돌리면 bbox가 대부분 겹쳤고(70-90은 98.9%가 IoU 0.5 이상), 75°와 90°는 같은 배치에서 조금 밀린 정도였다. 사진 단위로 나누면 val 사진과 거의 같은 배치의 사진을 train에서 본 셈이 되어, val 점수가 처음 보는 조합에서의 성능보다 높게 나올 수 있다(데이터 누수).
- 180° 회전 증강을 쓰면 train의 70° 사진이 val의 75°, 90° 사진과 더 비슷해진다.
- 두 방법으로 실제로 나눠 보고 다음을 센다. val 비율은 20%이고, 무작위로 나누는 결과가 seed에 따라 달라서 seed를 20번 바꿔 평균, 최소, 최대를 본다.
    - 같은 조합이 train에 있는 val 사진 비율: 누수가 얼마나 생기는지
    - val에만 있는 클래스 수: 학습하지 못하는 클래스
    - train에만 있는 클래스 수: val로 평가하지 못하는 클래스
- 조합 단위는 조합을 무작위 순서로 val에 넣다가 val 사진이 20%(46장) 이상이 되면 멈춘다. 마지막에 넣은 조합의 사진 수에 따라 val 사진 수가 46장보다 조금 많을 수 있다.
- 1차 학습에 쓴 분할은 8-2에 정리한다.

In [ ]:
VAL_RATIO = 0.2
N_SPLITS = 20  # seed를 바꿔 가며 나눠 보는 횟수

classes_by_photo = ann_df.groupby("file_name")["category_id"].agg(set)  # 사진마다 들어 있는 클래스
n_val_target = round(len(photo_df) * VAL_RATIO)


def split_by_photo(rng):
    """사진을 무작위로 n_val_target장 골라 val로 한다. val 사진 이름의 집합을 돌려준다."""
    val_idx = rng.choice(len(photo_df), n_val_target, replace=False)
    return set(photo_df["file_name"].iloc[val_idx])


def split_by_combo(rng):
    """조합을 무작위 순서로 val에 넣다가 val 사진이 n_val_target장 이상이 되면 멈춘다. 같은 조합의 사진은 함께 움직인다."""
    val_names = set()
    for combo in rng.permutation(combo_df.index.to_numpy()):
        if len(val_names) >= n_val_target:
            break
        val_names |= set(photo_df.loc[photo_df["combo"] == combo, "file_name"])
    return val_names


def split_stats(val_names):
    """val 사진 집합으로 누수 비율과 한쪽에만 있는 클래스 수를 센다."""
    is_val = photo_df["file_name"].isin(val_names)
    train_combos = set(photo_df.loc[~is_val, "combo"])
    leak = photo_df.loc[is_val, "combo"].isin(train_combos)
    train_cls = set().union(*classes_by_photo[photo_df.loc[~is_val, "file_name"]])
    val_cls = set().union(*classes_by_photo[photo_df.loc[is_val, "file_name"]])
    return {"val 사진 수": int(is_val.sum()),
            "같은 조합이 train에 있는 val 사진 비율": float(leak.mean()),
            "val에만 있는 클래스 수": len(val_cls - train_cls),
            "train에만 있는 클래스 수": len(train_cls - val_cls)}


split_rows = []
for k in range(N_SPLITS):
    split_rows.append({"분할": "사진 단위", **split_stats(split_by_photo(np.random.default_rng(SEED + k)))})
    split_rows.append({"분할": "조합 단위", **split_stats(split_by_combo(np.random.default_rng(SEED + k)))})
split_df = pd.DataFrame(split_rows)

print(f"val 목표: {n_val_target}장 (사진 {len(photo_df)}장의 {VAL_RATIO:.0%}), 반복: {N_SPLITS}번")
display(split_df.groupby("분할", sort=False).agg(["mean", "min", "max"]).round(3))

#### 6-4 결과

| 분할 단위 | val 사진 수 | 같은 조합이 train에 있는 val 사진 비율 | val에만 있는 클래스 수 | train에만 있는 클래스 수 |
|---|---|---|---|---|
| 사진 단위 | 46 | 평균 76.1% (63~87%) | 평균 0.1 (0~2) | 평균 11.6 (6~20) |
| 조합 단위 | 평균 46.7 (46~48) | 0% | 평균 4.1 (1~8) | 평균 24.8 (16~31) |

- 사진 단위로 나누면 val 사진의 약 4분의 3이 같은 조합의 다른 각도 사진을 train에 두고 있다. 6-3에서 같은 조합의 사진은 거의 같은 배치였으므로, val 점수가 처음 보는 조합에서의 성능보다 높게 나올 수 있다.
- 조합 단위로 나누면 누수는 없다. 하지만 두 가지 문제가 있다.
    - 평균 4.1개(최대 8개) 클래스가 val에만 들어가서 학습하지 못한다.
    - 평균 24.8개 클래스가 val에 없어서, 56개 클래스 중 절반 가까이를 val로 평가하지 못한다.
- 사진 단위로 나눠도 평균 11.6개 클래스는 val에 없다. 사진이 232장뿐이라, val 20%로는 어느 방법이든 모든 클래스를 평가하기 어렵다.
- 이 비교는 조합을 무작위로 고른 단순한 방법이다. 조합이 1개뿐인 클래스를 train에 고정하거나, 조합 단위로 여러 번 나눠 번갈아 평가(교차 검증)하면 결과가 달라질 수 있다.

### 6-5. 정리

#### 확인한 것

- 조합과 사진 수
    - 조합 114개의 사진 수는 1~3장이고, 같은 조합에 같은 각도의 사진은 한 장뿐이다.
    - 알약 4개짜리 조합 25개는 모두 사진이 3장이다. 알약 3개짜리 조합은 89개 중 74개가 3장보다 적다.
    - 뒷면 사진 4장은 조합 2개에 있고, 두 조합에는 앞면 사진이 없다.
- 같은 조합의 사진 관계
    - 75°와 90°는 같은 배치이고, 70°는 180° 돌린 배치다. 6-2 그림에서 보였고, 6-3에서 bbox 좌표로 전체 조합에서 확인했다.
    - 75° 사진은 다른 두 각도보다 bbox가 왼쪽 위로 밀려 있다. 그래서 75-90 쌍도 IoU 중앙값이 0.526에 그친다.
    - 같은 조합의 각도 쌍에서 IoU가 아주 낮은 짝 9쌍은 모두 3-8의 오류였고, 새로 찾은 오류는 없다.
- 분할 단위 (6-4, val 20%, seed 20번)
    - 사진 단위로 나누면 val 사진의 평균 76.1%가 같은 조합의 다른 각도 사진을 train에 두고 있다. 같은 조합의 사진은 거의 같은 배치라서 데이터 누수가 생긴다.
    - 조합 단위로 나누면 누수는 없지만, 평균 4.1개 클래스가 val에만 들어가 학습하지 못하고, 평균 24.8개 클래스는 val에 없어 평가하지 못한다.
    - 사진 단위로 나눠도 평균 11.6개 클래스는 val에 없다.

#### 정할 것 (이 EDA에서는 정하지 않는다)

- train/val 분할 단위: 사진 단위와 조합 단위 중 무엇을 쓸지 정한다. 조합 단위라면 조합이 1개뿐인 클래스를 train에 고정할지, 조합 단위 교차 검증으로 여러 번 나눠 평가할지도 함께 정한다. 1차 학습에 쓴 분할은 8-2에 정리한다.
- 증강: 180° 회전 증강을 쓰면 70° 사진이 75°, 90° 사진과 더 비슷해진다. 분할 단위와 함께 정한다.

#### 이 EDA에서 확인하지 않은 것

- 알약 3개짜리 조합 중 사진이 3장보다 적은 조합의 빠진 각도 사진이 test에 있는지는 확인하지 않았다.

## 7. test 데이터

- 대회 설명(1-1)에 test 데이터에는 train 데이터에 없는 클래스가 있다고 되어 있다.
- test 사진에는 라벨이 없어서, 어떤 클래스가 몇 개 있는지는 숫자로 셀 수 없다.
- test 사진을 눈으로 살펴본 결과, train 데이터에 없는 클래스(4-4의 클래스 그림에 없는 알약)가 많이 보였다.

#### 한계

- 검출 모델은 학습한 클래스만 찾아 분류할 수 있다. 평가 지표(mAP)는 클래스까지 맞아야 맞은 것으로 센다.
- 그래서 지금 제공된 train 데이터(56개 클래스)만으로는 test에만 있는 클래스를 학습할 수 없고, 그 알약은 맞힐 수 없다.
- 이 클래스들을 학습하려면 AI Hub 등에서 데이터를 추가해야 한다. 대회 규칙에 따라 AI Hub의 `TL_2_조합.zip`, `TS_2_조합.zip`은 쓸 수 없다.

## 8. 1차 학습에 적용한 라벨 수정과 train/val 분할

이 EDA를 바탕으로 1차 학습에서 실제로 쓴 라벨 수정과 train/val 분할을 정리한다.

### 8-1. 라벨 수정

- 1차 학습에는 아래 표의 라벨 수정을 적용했다.
- 수정은 21개다. 새로 추가한 bbox가 8개, 고친 bbox가 13개이고, 3-8의 오류 21개와 같은 bbox다. 원본 JSON 763개에 적용하면 JSON이 771개가 된다.
- 3-8의 방안과 다르게 적용한 것이 하나 있다. `K-003351-019232-035206` 90°의 19232는 3-6에서 bbox 너비 322 → 332 후보가 나왔지만, 3-8에서는 area만 근거라 사진을 다시 보고 정하기로 했다. 수정 목록에서는 bbox를 그대로 두고 area를 너비 × 높이(322 × 266 = 85652)로 고쳤다. 그래서 아래 표에서는 이 bbox가 area 값 수정에 들어간다.

| 종류 | 사진 | 클래스 | 고친 값 | 수정 전 | 수정 후 |
|---|---|---|---|---|---|
| 라벨 누락 | `K-003351-013900-021325` 70° | 3351 | bbox 추가 | 없음 | `[385, 836, 188, 194]` |
| 라벨 누락 | `K-003351-013900-036637` 70° | 3351 | bbox 추가 | 없음 | `[430, 866, 188, 194]` |
| 라벨 누락 | `K-003351-020014-022074` 90° | 20014 | bbox 추가 | 없음 | `[50, 709, 348, 340]` |
| 라벨 누락 | `K-003351-021325-032310` 90° | 32310 | bbox 추가 | 없음 | `[579, 816, 377, 250]` |
| 라벨 누락 | `K-003351-032310-038162` 70° | 3351 | bbox 추가 | 없음 | `[380, 857, 191, 194]` |
| 라벨 누락 | `K-003351-033880-038162` 75° | 33880 | bbox 추가 | 없음 | `[60, 586, 315, 438]` |
| 라벨 누락 | `K-003351-035206-041768` 70° | 3351 | bbox 추가 | 없음 | `[454, 873, 188, 195]` |
| 라벨 누락 | `K-003544-004543-012247-016548` 90° | 4543 | bbox 추가 | 없음 | `[634, 194, 207, 203]` |
| 이미지 밖 bbox | `K-003351-016262-018357` 75° | 18357 | bbox | `[6567, 625, 311, 315]` | `[567, 625, 311, 315]` |
| 다른 알약 위 bbox | `K-001900-016548-019607-033009` 70° | 16548 | bbox | `[88, 255, 366, 209]` | `[99, 862, 245, 239]` |
| 다른 알약 위 bbox | `K-003351-020238-031863` 70° | 20238 | bbox | `[446, 844, 190, 191]` | `[585, 292, 223, 220]` |
| 다른 알약 위 bbox | `K-003351-029667-031863` 70° | 3351 | bbox | `[178, 249, 205, 203]` | `[372, 856, 188, 193]` |
| 빈 바닥 bbox | `K-003351-020014-020238` 90° | 3351 | bbox | `[185, 249, 188, 183]` | `[385, 249, 188, 183]` |
| 클래스 바뀜 | `K-003483-025367-027733-035206` 75° | 27733 | bbox | `[565, 119, 257, 380]` | `[87, 726, 321, 283]` |
| 클래스 바뀜 | `K-003483-025367-027733-035206` 75° | 35206 | bbox | `[87, 726, 321, 283]` | `[565, 119, 257, 380]` |
| bbox 크기 | `K-003351-019232-029667` 70° | 19232 | bbox | `[523, 201, 232, 241]` | `[523, 201, 232, 341]` |
| bbox 크기 | `K-003351-021325-033880` 70° | 3351 | bbox | `[427, 839, 189, 182]` | `[427, 839, 189, 192]` |
| area 값 | `K-003351-019232-035206` 90° | 19232 | area | 88312 | 85652 |
| area 값 | `K-003351-033880-038162` 70° | 33880 | area | 129480 | 129780 |
| area 값 | `K-003351-033880-038162` 70° | 38162 | area | 62009 | 65009 |
| area 값 | `K-002483-012081-012778-025438` 90° | 12778 | area | 161096 | 161036 |

| 종류 | 수 |
|---|---|
| 라벨 누락 | 8 |
| 이미지 밖 bbox | 1 |
| 다른 알약 위 bbox | 3 |
| 빈 바닥 bbox | 1 |
| 클래스 바뀜 | 2 |
| bbox 크기 | 2 |
| area 값 | 4 |
| 합계 | 21 |

### 8-2. train/val 분할

- 회의에서 고정한 EDA 분할(seed 410)을 그대로 썼다.
- 분할 단위는 조합이다. 같은 조합의 사진은 모두 train이나 val 한쪽에 들어가서, 6-4에서 본 데이터 누수(같은 조합의 다른 각도 사진이 train과 val에 나뉘는 것)가 생기지 않는다.
- 조합이 모자라서 train과 val로 나눌 수 없는 조합은 train에 넣었다.

| 구분 | 사진 수 |
|---|---|
| train | 185 |
| val | 47 |
| 합계 | 232 |

- 분할 정보가 빠진 bbox는 0개다. 모든 bbox가 train이나 val 중 하나에 들어 있다.